Useful links:

Model Leaderboard: https://artificialanalysis.ai/leaderboards/models


In [1]:
# Imports
from openai import OpenAI
import os
from typing import List, Dict, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
from huggingface_hub import snapshot_download
from mlx_lm import load, generate

In [2]:
# Globals
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SYS_PROMPT = (
        "You are an AI research assistant for a RAG pipeline. "
        "You will receive a user query and a set of plain text research documents. "
        "Your tasks: (1) explain overall relevance of the documents to the query, "
        "(2) summarize each document briefly, and (3) answer the query if it is a question. "
        "CRITICAL RULES:\n"
        "- Rely ONLY on the provided documents; if something isn't supported, say so.\n"
        "- After claims, cite sources using bracketed ids like [paperA] or [paperB].\n"
        "- Keep writing concise and factual; avoid hype.\n"
        "- Output EXACTLY three sections with the headings below."
    )

In [3]:
# Test Queries

docs = [
    {
        "id": "paper1",
        "title": "Ai2 Scholar QA: Organized Literature Synthesis with Attribution",
        "abstract": """Retrieval-augmented generation is increasingly
                    effective in answering scientific questions from
                    literature, but many state-of-the-art systems are
                    expensive and closed-source. We introduce Ai2
                    Scholar QA, a free online scientific question
                    answering application. To facilitate research,
                    we make our entire pipeline public: as a cus-
                    tomizable open-source Python package1 and
                    interactive web app, along with paper indexes
                    accessible through public APIs and download-
                    able datasets. We describe our system in detail
                    and present experiments analyzing its key de-
                    sign decisions. In an evaluation on a recent sci-
                    entific QA benchmark, we find that Ai2 Scholar
                    QA outperforms competing systems.""",
        "text": """arXiv:2504.10861v2 [cs.CL] 28 Jul 2025
: Organized Literature Synthesis with Attribution
Amanpreet Singh* Joseph Chee Chang∗ Chloe Anastasiades∗ Dany Haddad∗
Aakanksha Naik Amber Tanaka Angele Zamarron Cecile Nguyen Jena D. Hwang
Jason Dunkleberger Matt Latzke Smita Rao Jaron Lochner Rob Evans
Rodney Kinney Daniel S. Weld Doug Downey∗ Sergey Feldman∗
Allen Institute for AI
{amanpreets, sergey}@allenai.org
Abstract
Retrieval-augmented generation is increasingly
effective in answering scientific questions from
literature, but many state-of-the-art systems are
expensive and closed-source. We introduce Ai2
Scholar QA, a free online scientific question
answering application. To facilitate research,
we make our entire pipeline public: as a cus-
tomizable open-source Python package1 and
interactive web app, along with paper indexes
accessible through public APIs and download-
able datasets. We describe our system in detail
and present experiments analyzing its key de-
sign decisions. In an evaluation on a recent sci-
entific QA benchmark, we find that Ai2 Scholar
QA outperforms competing systems.
qa.allen.ai
allenai/ai2-scholarqa-lib
Demo Video
Python Package
1 Introduction
Long-form scientific question answering systems
use retrieval-augmented generation (RAG) (Lewis
et al., 2020) over scientific literature to answer com-
plex questions. These systems produce responses
that bring together relevant insights from dozens of
papers to help users rapidly learn about a body of
scientific work. Examples are OpenScholar (Asai
et al., 2024), Elicit, Consensus, and others §5.
Most of these systems are expensive to use and
closed source, relying on models, workflows, and
retrieval solutions not shared publicly. These issues
create barriers for researchers who wish to study
or build on the work. In response, we introduce
Ai2 Scholar QA, a free-to-use scientific QA system
(qa.allen.ai), and share our key components as
open source software and public APIs.
Scholar QA follows a multi-stage pipeline (Fig-
ure 1) that starts by querying paper indexes: one
* Core contributors
1We use closed state-of-the-art LLMs.
from Semantic Scholar with over 100M abstracts,
and a new index that we introduce in this work
containing 11.7M full-text scientific papers. The
pipeline then re-ranks the retrieved passages with
a cross-encoder, and finally prompts a Large Lan-
guage Model (LLM) to filter, cluster, and synthe-
size the passages into an answer. The final answer
is presented to the user in a report with expand-
able sections of prose, bulleted lists, and tables.
Claims in the answer are supported by citations,
which can be clicked to reveal the cited paper’s
title and authors (with links to their corresponding
Semantic Scholar pages), and in many cases rele-
vant excerpt(s) from the paper, allowing for quick
verification of the claim.
The system is based on open source code, en-
abling the community to reproduce and build on
it. We release the code for our pipeline, prompt-
ing workflow and Web application. The retrieval
indexes, including the new full text search index,
are available as Semantic Scholar APIs and dataset
downloads, and are continually updated with new
articles (Kinney et al., 2023). Together, these re-
sources can be combined with any generative LLM
API to power a complete long-form scientific QA
application. Our production system currently uses
Anthropic’s Claude 3.7 (Anthropic, 2024).
We present analyses that justify key design de-
cisions in our architecture in §4. Our choice of
retrieval models and configuration is informed by
evaluation over a collection of real and synthetic
user queries and accompanying passages judged for
relevance by a LLM, both of which we release pub-
licly. We compare Scholar QA’s answers against
several baselines, demonstrating that it achieves
state-of-the-art performance on the ScholarQA-CS
benchmark (Asai et al., 2024). Finally, we discuss
the reception of Scholar QA by users. The strong
majority (85%) of user feedback is positive, and the
reported issues suggest important improvements
for future work.
OUTPUT
Long-form answer report
Query Decomposer
• Query rephrase, dense & sparse
• Paper filters
“ ”
11.7M
full-text
papers
Query Validation
INPUT User Query ScholarQA
with omni-moderation-latest
…
Per paper,
aggregate
passages
Per paper,
per passage,
extract quotes
“ ”
…
Step 1 Search APIs:
Top 50-ranked
Passages
“ ”
“
Passage search
“ ”
Build answer outline & synthesize
extracted quotes to outline. E.g.,
Background
section
Bulleted
sections
Summary
paragraphs
:
•
•
“ ”
“
”
“ ” “ ”
+
Keyword search (100M papers)
Cross-Encoder
Reranker
”
w/ LLM
Extract quotes from passages
w/ LLM
Cluster Extracted Quotes
w/ LLM
Step 3
Report Generation w/ Tables
Retrieval (§2.1) Reranker (§2.2)
2 Pipeline
The Scholar QA architecture (Figure 1) has three
primary components: 1) retrieval to identify rel-
evant passages from a corpus of scientific litera-
ture; 2) a neural cross-encoder that re-ranks the
passages to select the most relevant top-k; and 3)
multi-step LLM generation to process the passages
into a comprehensive report. Next, we describe
each component of the pipeline in detail.
Query Validation. Prior to processing a query,
we employ OpenAI’s omni-moderation-latest2
Step 3
w/ LLM
model for safeguarding against potentially harmful
content and return appropriate error messages.
2.1 Retrieval
We use the Semantic Scholar API (Kinney et al.,
2023) for retrieval, specifically its endpoint for key-
word search over paper abstracts, and our new end-
point for querying snippets from open-access pa-
pers. A query decomposer re-formulates the user
query for each endpoint and retrieves up to 256
snippets and 20 abstracts. These texts are referred
to as "passages" below.
Query Decomposer. The two retrieval endpoints
differ in their effective query formats (one targets
keyword and the other semantic queries) and filter-
ing of results based on the user’s preferences for pa-
per metadata (paper year, venue, field of study). In
our query decomposition step, an LLM is prompted
to re-format the user query into paraphrases appro-
priate for each endpoint, and to extract the user’s
requested settings for the metadata filters. We use
the outputs of this step for retrieval.
Search APIs. The Semantic Scholar keyword
search API is described in Kinney et al. (2023). We
introduce a new /snippet/search endpoint, which
searches over a corpus of passages extracted from
S2ORC (Lo et al., 2020), loaded into a Vespa clus-
ter with papers and passages. Papers include meta-
data for filtering. Passages are derived from a pa-
2https://platform.openai.com/docs/guides/
moderation
Step 2 Multi-Step Generation (§2.3)
Figure 1: Scholar QA Pipeline Overview
per’s title, abstract, or body and can be filtered at
the paper level. The index includes 11.7M full-text
papers across the fields of study listed here, and a
total of 285.6M passages.
Each passage is limited to 480 tokens and trun-
cated at sentence and section boundaries where
possible, having an overlap of one sentence (up
to 64 tokens) with the preceding and follow-
ing passages. Passage text is embedded with
mxbai-embed-large-v1 (Lee et al., 2024) with
binary quantization, and placed into a dense (ap-
Step 2
w/ LLM
proximate nearest neighbor) index, as well as a
traditional sparse keyword index.
We first retrieve a union of embedding and
keyword-based matches, applying any specified fil-
ters. The filtered results are ranked with a weighted
sum of embedding similarity and bm25 scores.
2.2 Reranking
The passages obtained from the retrieval step are
subsequently passed to a neural re-ranker and the
top 50 results are retained. The re-ranker is a
cross-encoder that encodes both the query and a
candidate document simultaneously and outputs a
relevance score used to rank the documents. We
selected mxbai-rerank-large-v1 (Shakir et al.,
2024) based on the results in §4.2 and host it on
Modal with a single NVIDIA L40S GPU.
2.3 Multi-step Generation
The generation phase employs a three-step ap-
proach: first, the retrieved passages are processed
to extract more precise quotes relevant to the query;
second, the quotes are thematically clustered into
separate sections appropriate for the answer; finally,
a controlled generation process composes the final
report one section at a time, synthesizing the quotes
assigned to that section.
Quote extraction. Passages from the retrieval
stage can be lengthy and may contain extraneous
information not useful for answering the user query
(Asai et al., 2023). The quote extraction stage aims
to select only the most relevant quotes from the
passages to improve the precision of the answer.
We instruct an LLM to extract verbatim quotes
that directly contribute to answering the query (Slo-
bodkin et al., 2024). As input to the extraction, we
gather all passages from the re-ranker for a given
paper, and concatenate these to the abstract of the
paper. This aggregation helps create a richer con-
text conducive to extracting relevant quotes. The
LLM processes each paper’s content independently
and returns the selected quotes separated by el-
lipses. If the entire paper context is deemed irrele-
vant, it is discarded from further processing.
Answer Outline and Clustering. For generating
a comprehensive research report, the effective or-
ganization of reference materials is essential for its
overall coherence. We propose a thematic outline
framework where the answer is divided into sec-
tions representing topics, and the reference quotes
are assigned to these topics. This mapping allows
the system to selectively focus only on the pertinent
subset of quotes when synthesizing a section.
First, the LLM is instructed to generate a list of
themes in logical order and the appropriate syn-
thesis format for each theme, independent of the
quotes from the previous step. The first section is al-
ways an introduction or background to provide the
user the basics for understanding the answer. The
format of each section can be either a paragraph or
a bulleted list, serving different information needs.
Paragraphs convey nuanced summaries from multi-
ple papers, while bulleted lists enumerate related
papers (e.g., models, datasets, or interactive sys-
tems). These list are also the catalyst for generating
the comparison tables (see §2.3). Following this,
the sections are assigned 0 or more quotes. In case
no quote is assigned to a section, it is generated
completely from the LLM weights.
Report Generation. With the answer outline in
place, each section of the report is synthesized se-
rially conditioned on the query, reference sources,
and the sections prior to it. The LLM is also in-
structed to generate a TLDR for each section. The
references are either the quotes assigned to the sec-
tion or abstracts of papers that are cited within these
quotes. This citation following method allows the
LLM to condition on and cite foundational sources
which are not uncovered in retrieval. The LLM is
instructed to cite the sources for each claim in the
generated section text and cite generations from its
parameters as LLM Memory.
Paper Comparison Table Generation. Since bul-
leted list sections typically include closely related
papers (e.g., different datasets), we additionally
generate tables that compare and contrast all pa-
pers cited in that section using common aspects
(e.g., size and annotation method). This pipeline is
detailed in Newman et al. (2024). At a high level,
the inputs are the query to Scholar QA, the section
title, and the abstracts of all papers cited in the
section. An LLM first produces a set of common
aspects (columns) to compare papers (rows). Each
cell (paper-aspect pair) is filled with a value using
the full-text of the paper. Finally, as not all aspects
are applicable to every paper (e.g., one paper might
not be about a dataset), we filter out columns and
rows with a high proportion of missing values. Fig-
ure 3 [A] shows an expanded table in Scholar QA
where related papers from a section are compared
across a set of common aspects ([B]).
3 Scholar QA: Interface and Source Code
Scholar QA is open-sourced as an extensible
Python package (ai2-scholar-qa) and a Type-
script and React-based interactive web application.
The LLM functionality of Scholar QA is imple-
mented with litellm, which supports swapping a
variety of models using your own keys. Thus, the
community can build upon Scholar QA and easily
visualize the results (examples in Appendix A). Be-
low we describe the user experience of the demo.3
Progress and Section Streaming. High system
latency can hinder usability. On average, Scholar
QA produces a full report in 2.5 minutes (N=500,
σ=70s), which is comparable to modern LLM-
based research tools. To further improve usability,
the following designs were used: 1) Displaying
detailed real-time progress of the system (Nielsen,
1994) so users can examine the number of papers,
passages, and sections being processed. 2) Present-
ing each section as soon as it is generated, so users
can begin browsing the first section in 50 seconds
(N=500, σ=24s) post issuing a query (Appendix H).
Expandable Sections. By default, sections are
collapsed showing only their titles, TLDR sum-
maries, and number of cited sources. This gives
users a gist of the information included in the re-
port (Figure 2 [A]). Users can then click on the title
of a section they wish to read to expand it ([B]).
3Our production system has a few additional features like
downloadable reports, login and links to other Ai2 systems.
Figure 2: Multi-section [B] report generated by Scholar QA. References are linked to supporting excerpts [C].
Thumbs and free text feedback are collected for the full report [A], and also for each section and inline table.
References and Evidence Excerpts. To verify
the claims in the report, users can click on the inline
citations (Figure 2 [C]) or the pink excerpt icon in
the inline table cells (Figure 3 [C]) to bring up a
popup paper card. From the paper card, they can
see the relevant excerpts used during the generation
or click on the title to open the paper directly.
User Feedback Collection. We collect thumbs
up/down or textual feedback for the whole report
(Figure 2 [A]) and at each section and inline table.
4 Evaluation
4.1 Retrieval
We tuned our retrieval setup by optimizing rank-
ing over a dev set of 500 synthetic queries (see
Appendix C) and the top 1000 passages for each
based on GIST embedding distance (Solatorio,
2024). We generated binary relevance labels with
gpt-4-turbo (see Appendix B for the prompt),
which were found to have 80% agreement with
Figure 3: Inline tables compare papers [A] with com-
mon aspects [B] with values linked to supporting ex-
cerpts from the papers [C].
Figure 4: Embedding ranking performance for various
compression methods and matryoshka cutoffs. The x-
axis indicates the size of the vector index based relative
to using int8 quantization and the full embedding size.
The red circle indicates the selected configuration. Em-
bedding size is notated next to each point.
human annotators on a sample of 100 queries.
Pipeline Tuning. We optimized several aspects
of retrieval over this dev set: embedding model
selection and quantization method for it, the com-
ponents and weights in the final ensemble, and
(when relevant) the target Matryoshka dimension
for the embeddings (Kusupati et al., 2024).
We experimented with medium sized embedding
models based on top performers on the retriever
and ranking tasks of the MTEB (Muennighoff
et al., 2022) leaderboard on HuggingFace. Table 4
in Appendix D lists our candidate models. The
mxbai-embed-large-v1 (Lee et al., 2024) embed-
dings performed best over our dev set. Figure 4 val-
idates our choice of quantization method and target
Matryoshka dimension for these embeddings. We
chose ubinary quantization with no Matryoshka
truncation, (indicated by a red circle on the plot)
since it satisfied our storage constraints without a
large drop in performance. We experimented with
ensembling SparseEmbed (Kong et al., 2023), em-
bedding cosine similarity, BM25, and chose the
latter two (weight split of (0.6,0.4) respectively)
based on the results (See Appendix E). The BM25
scores are normalized with min-max scaling before
computing the ensemble score.
4.2 Reranking
We chose the re-ranker based on evaluation over a
mixture of real scientific questions from the Stack
Exchange Computer Science, Math, and Statistics
communities, real research queries written by the
authors and their colleagues, and synthetic ones
generated by fine-tuning GPT-4o-mini over ques-
tions from the ScholarQA-CS dataset (Asai et al.,
Model (Size) Latency
(sec/query)
nDCG
@10 mRR
bge-reranker-v2-m3 (568M) 0.14 0.913 0.973
akariasai/ranker_large (568M) 0.14 0.906 0.970
jina-reranker-v2-base (278M) 0.06 0.907 0.972
mxbai-rerank-large-v1 (435M) 0.46 0.927 0.975
mxbai-rerank-base-v1 (184M) 0.19 0.919 0.974
mxbai-rerank-xsmall-v1 (70M) 0.11 0.911 0.970
mxbai-rerank-base-v2 (0.5B) 0.40 0.918 0.974
mxbai-rerank-large-v2 (1.5B) 0.70 0.911 0.975
Table 1: Cross encoder re-ranker results on our dataset
of GPT-4o labels. The best results are highlighted.
2024). For a given query, passages are retrieved
and then awarded a relevance score in the range
0-3 with GPT-4o. We experiment with multiple
state-of-the-art re-rankers (Chen et al., 2024; Shakir
et al., 2024; Asai et al., 2024), and, as shown in
Table 2, mxbai-rerank-large-v1 gives the best
results across the board (even outperforming its v2
model on our task). To reduce latency for deploy-
ment, we implemented optimizations like Pytorch
model compilation. We release the evaluation data
consisting of 2,426 queries and 225,618 passages.
4.3 Generation
We evaluate the final output of Scholar QA on the
ScholarQA-CS dataset which consists of expert-
annotated rubrics for 100 Computer Science re-
search questions. The question-specific expert
rubrics account for 60% of the final score, while
the rest is computed based on global metrics of
length, expertise and citations. We use GPT-4o
(Hurst et al., 2024) as a judge with the utility pro-
vided by Asai et al. (2024) for automatic evaluation
and compare against several baselines.
As shown in Table 2, our system outperforms
popular LLMs: Llama 3.1 (Dubey et al., 2024),
GPT 4.1 and Claude Sonnet 3.7 (Anthropic, 2024).
It even outperforms reasoning models such as Son-
net 3.7 Thinking (Anthropic, 2025), o1-mini (Ope-
nAI, 2024b) and o3-mini (Zhang et al., 2025) over-
all on the Scholar QA-CS benchmark. This setup
lacks any retrieval so the models generate the re-
sponses completely from parametric memory. The
benchmark rewards attribution and supporting ev-
idence as a measure of trust in the system, so
these models score lower overall. The reasoning
based models perform better than our system on
the rubrics score, which suggests that they may be
superior backbones for our system. However, due
to the additional reasoning tokens, these models
are more expensive and also significantly increase
latency.
For contemporary QA systems, we compare
against OpenScholar with GPT-4o4, PaperQA2
(Skarlinski et al., 2024), Perplexity’s Sonar Deep
Research and STORM (Shao et al., 2024a). Pa-
perQA2 did not release their retrieval corpus, so
we substitute it with our retrieval pipeline for a fair
comparison. Scholar QA obtains the best scores
both on rubrics and overall, with the variant us-
ing Claude 3.7 Sonnet as the backbone scoring 2.4
points higher than STORM. For these QA systems,
we also evaluate the attribution quality based on
ALCE (Gao et al., 2023), which proposes entail-
ment between claims and evidence to compute ci-
tation precision and recall. Again, we use GPT-4o
as a judge to predict entailment (See Appendix F
for the prompt) and treat each sentence in a re-
sponse as a claim. Even with a report spanning
multiple sections where all the sentences might not
be cited, Scholar QA comes out far ahead of the
other QA systems. Due to a lack of retrieval, this
evaluation was not conducted when the LLMs are
simply prompted to generate a response from mem-
ory. An interesting discovery from our analysis
was that with an updated version of GPT-4o (i.e.
gpt-4o-2024-11-20) as the judge, the scores are
inflated compared to using gpt-4o-2024-08-06,
even though the relative rankings are consistent
(See Appendix J). For parity with Asai et al. (2023),
we report the rubrics and citation scores with the
older and newer model as the judge, respectively.
During our initial experiments, we restricted
ScholarQA to only summarize the insights con-
ditioned on the quotes extracted from retrieved pas-
sages. However, in cases where the retrieved pas-
sages were not relevant enough, the system failed
to answer the question in favor of just discussing
the information in the quotes. Moreover, for over
30% of instances in ScholarQA-CS, the rubrics
require background information, even though the
question might not. So, we updated our system
LLM prompts to – a) Generate section text from
memory if there is a lack of relevant retrieved pas-
sages and cite as LLM Memory and b) generate
the first section as a background or introduction for
the rest of the answer. The results reported here are
obtained post these changes.
To finalize the backbone LLM for the production
web application we conducted an anonymized pair-
4Our results are not identical to Asai et al. (2024) due to
variance across LLM-as-a-judge runs. Their reported total
score for OS-GPT-4o is 57.7. We re-ran the evaluation in
order to obtain rubrics only scores, which they did not report.
Model Score Model Score
Rubrics Total Rubrics Total Cite
LLM Prompting (No Retrieval) QA Systems
Llama 3.1-8B 48.8 47.3 Llama 3.1-70B 52.4 48.6 Claude 3.5 S 50.4 46.6 Claude 3.7 S 61.5 55.9 +Thinking 62.7 55.7 GPT-4.1 63.2 56.2 SQA-Claude 3.7 S 58.0 61.9 48.1
SQA-Claude 3.5 S 52.6 61.3 52.1
OS-GPT-4o 49.3 53.5 25.9
PaperQA2 38.7 51.4 25.3
Perplex. Sonar DR 38.7 52.8 25.2
STORM 54.2 59.5 40.2
o1-mini 62.3 55.5
o3-mini 60.6 50.2
Table 2: Evaluation results on ScholarQA-CS bench-
mark. System responses are either generated by simply
prompting LLMs with the questions or by issuing the
queries to RAG based QA systems. Expert annotated
rubrics only scores are reported in addition to the over-
all total. The overall best results are highlighted and
best results within a category are underlined. SQA: Ai2
Scholar QA, OS: Open Scholar, S: Sonnet, Claude 3.5
S: claude-3-5-sonnet-20241022.
wise comparison among the authors of this work.
We compare Claude 3.7 against 3.5. Out of 18 com-
parisons, Claude 3.7 Sonnet was the overwhelming
favorite with 17 wins, reinforcing our hypothesis
that (with no other changes) our system improves
with newer and better backbone LLMs.
4.4 Real-world Usage and User Feedback
We have publicly deployed Scholar QA for 9
weeks, and received 30.2k questions from 8,219
unique visitors. On average, each response is about
2.4k words and costs $0.50 to produce. We ob-
served 1,075 monthly repeated users who had is-
sued queries on two distinct days over the course of
a 30 day window. We analyze the user query types
and the most prominent themes were deep-dive
into specific research topics (15k) and comparative
analysis of specific prior work (5k) (detailed dis-
tribution in Appendix I). A total of 2,433 thumbs
feedback were submitted (Figure 2 [A]) and 85%
were positive. These suggests real-world users ben-
efited from using Scholar QA.
For insight into the failure modes, we manually
examined the 383 instances of neutral/negative free-
form feedback. Table 3 lists the feedback types we
identified along with their counts as of May 2025
(example feedback in Appendix G). We hypoth-
esize that follow-up questions may help address
insufficient answer detail and cases with a lack of
retrieved documents, while improved retrieval may
help address incomplete or incorrect references and
off-topic responses.
Category Count
Incorrect or Missing References 126
Off-topic or Misunderstood Query 113
Request for More Detail or Specificity 289
General Feedback on Quality 149
Language or Format Issues 78
Table 3: Feedback Categories and Counts
5 Related Work
Scientific Question Answering. Answering sci-
entific questions involves navigating scholarly
sources and accurately retrieving and synthesizing
them. Recently, OpenScholar (Asai et al., 2024)
introduced a retrieval-augmented model designed
explicitly for scientific literature synthesis with
citation-supported responses with significant im-
provement in accuracy and reduced citation halluci-
nation. Scholar QA extends its capabilities by lever-
aging the latest state-of-the-art LLMs and an open
source generation pipeline that filters literature into
precise quotes and produces thematically organized
and detailed answers. STORM (Shao et al., 2024b)
synthesizes comprehensive, Wikipedia-like articles,
a distinct task from long-form scientific question
answering. Other works have focused on litera-
ture review synthesis: LitLLM (Agarwal et al.,
2024), which like Scholar QA uses a structured
planning-and-generation pipeline similar, and Sur-
veyForge (Yan et al., 2025), which outlines heuris-
tics before generation. Their code was not available
at the time of our evaluation. Zhou et al. (2025)
present a survey categorizing AI-driven research
support systems across various stages of the scien-
tific process, including literature synthesis.
Commercial Tools for Scientific QA. Commer-
cial RAG tools have emerged to facilitate research
specifically tailored for scientific literature, such
as Consensus (Consensus, 2024), which synthe-
sizes findings from research papers, Scite (Scite,
2024), which evaluates claims by analyzing cita-
tion contexts, and Elicit (Elicit, 2024), which sup-
ports structured scientific literature reviews. Other
general-purpose tools also support scientific in-
quiries: Perplexity (Perplexity, 2024), You.com
(You.com, 2024), OpenAI Deep Research (Ope-
nAI, 2024a) and Gemini Deep Research (Deep-
Mind, 2024). Although these platforms leverage
advanced retrieval and generation capabilities to fa-
cilitate literature reviews and deliver rapid insights,
they can be too expensive for widespread academic
use and typically lack transparency regarding their
pipelines. In contrast, Scholar QA is free with open
sourced code and access to search APIs that enable
the research community to build upon it.
6 Conclusion
We present Ai2 Scholar QA, a freely-available long-
form literature synthesis system that generates re-
ports for complex scientific questions. We release
key components as open source code and public
APIs, and report experiments analyzing design de-
cisions and demonstrate state-of-the-art results.
Limitations
Supplementing the user feedback discussed in sub-
section 4.4, we would like to outline some limita-
tions of our system and evaluation and our plans to
mitigate them as part of fuure work:
(i) Ai2 Scholar QA uses proprietary and closed-
source LLM as the backbone for our produc-
tion pipeline. As shown in Table 2, open
source models lag behind the proprietary mod-
els in our evaluation. However, we are actively
experimenting with open-sourced LLMs to re-
place the closed ones partially or completely
in the pipeline. The open-sourced models will
be specifically trained to do well on long-form
scientific question answering and each of the
sub-tasks in our multi-step generation. Fur-
ther, our code is open-sourced and can easily
be used with potentially any available LLM
api provider supported by litellm.
(ii) We evaluate the answers generated by Scholar
QA and compare against other systems on
ScholarQA-CS dataset in subsection 4.3.
Even though the answer rubrics are collected
via human annotation, the evaluation is only
limited to questions in the Computer Science
domain and further relies completely on an
LLM as the evaluator. In ongoing work, we
are investigating more accurate benchmarks
for evaluating long form scientific answers.
Our approach uses real queries posed by users
to Scholar QA, and human preference labels
over answers from multiple systems in not just
Computer Science, but Biomedicine and other
scientific domains. These labels can serve as
not only for evaluation, but also as training
signals for models.
Acknowledgments
We would like to thank the anonymous reviewers
for helpful comments, suggestions and feedback on
the manuscript. We would also like to acknowledge
the Ai2 ScholarQA users for providing construc-
tive feedback that helped us improve the system.
Finally, we thank David Albright for helping with
the demo video, the Ai2 communications team for
their help with user outreach, and Ai2 engineers
and researchers for their help with user testing be-
fore launch.
References
Shubham Agarwal, Gaurav Sahu, Abhay Puri, Is-
sam Hadj Laradji, Krishnamurthy Dj Dvijotham, Ja-
son Stanley, Laurent Charlin, and Christopher Pal.
2024. Litllms, llms for literature review: Are we
there yet?
Anthropic. 2024. The claude 3 model family: Opus,
sonnet, haiku.
Anthropic. 2025. Claude 3.7 sonnet system card.
Akari Asai, Jacqueline He, Rulin Shao, Weijia Shi,
Amanpreet Singh, Joseph Chee Chang, Kyle Lo,
Luca Soldaini, Sergey Feldman, Mike D’Arcy,
David Wadden, Matt Latzke, Minyang Tian, Pan
Ji, Shengyan Liu, Hao Tong, Bohao Wu, Yanyu
Xiong, Luke S. Zettlemoyer, and 6 others. 2024.
Openscholar: Synthesizing scientific literature with
retrieval-augmented lms. ArXiv, abs/2411.14199.
Akari Asai, Zeqiu Wu, Yizhong Wang, Avirup Sil, and
Hannaneh Hajishirzi. 2023. Self-rag: Learning to
retrieve, generate, and critique through self-reflection.
ArXiv, abs/2310.11511.
Jianlv Chen, Shitao Xiao, Peitian Zhang, Kun Luo, Defu
Lian, and Zheng Liu. 2024. Bge m3-embedding:
Multi-lingual, multi-functionality, multi-granularity
text embeddings through self-knowledge distillation.
Preprint, arXiv:2402.03216.
Consensus. 2024. Consensus – ai for research. Ac-
cessed: 2025-03-28.
Google DeepMind. 2024. Gemini – deep research mode.
Accessed: 2025-03-28.
Abhimanyu Dubey, Abhinav Jauhri, Abhinav Pandey,
Abhishek Kadian, Ahmad Al-Dahle, Aiesha Letman,
Akhil Mathur, Alan Schelten, Amy Yang, Angela
Fan, Anirudh Goyal, Anthony S. Hartshorn, Aobo
Yang, Archi Mitra, Archie Sravankumar, Artem Ko-
renev, Arthur Hinsvark, Arun Rao, Aston Zhang, and
510 others. 2024. The llama 3 herd of models. ArXiv,
abs/2407.21783.
Elicit. 2024. Elicit – the ai research assistant. Accessed:
2025-03-28.
Tianyu Gao, Howard Yen, Jiatong Yu, and Danqi Chen.
2023. Enabling large language models to generate
text with citations. In Conference on Empirical Meth-
ods in Natural Language Processing.
OpenAI Aaron Hurst, Adam Lerer, Adam P. Goucher,
Adam Perelman, Aditya Ramesh, Aidan Clark,
AJ Ostrow, Akila Welihinda, Alan Hayes, Alec Rad-
ford, Aleksander Mkadry, Alex Baker-Whitcomb,
Alex Beutel, Alex Borzunov, Alex Carney, Alex
Chow, Alexander Kirillov, Alex Nichol, Alex Paino,
and 397 others. 2024. Gpt-4o system card. ArXiv,
abs/2410.21276.
Rodney Michael Kinney, Chloe Anastasiades, Rus-
sell Authur, Iz Beltagy, Jonathan Bragg, Alexan-
dra Buraczynski, Isabel Cachola, Stefan Candra, Yo-
ganand Chandrasekhar, Arman Cohan, Miles Craw-
ford, Doug Downey, Jason Dunkelberger, Oren Et-
zioni, Rob Evans, Sergey Feldman, Joseph Gorney,
David W. Graham, F.Q. Hu, and 29 others. 2023.
The semantic scholar open data platform. ArXiv,
abs/2301.10140.
Weize Kong, Jeffrey M. Dudek, Cheng Li, Mingyang
Zhang, and Michael Bendersky. 2023. Sparseembed:
Learning sparse lexical representations with contex-
tual embeddings for retrieval. In Proceedings of the
46th International ACM SIGIR Conference on Re-
search and Development in Information Retrieval,
SIGIR ’23, page 2399–2403. ACM.
Aditya Kusupati, Gantavya Bhatt, Aniket Rege,
Matthew Wallingford, Aditya Sinha, Vivek Ra-
manujan, William Howard-Snyder, Kaifeng Chen,
Sham Kakade, Prateek Jain, and Ali Farhadi. 2024.
Matryoshka representation learning. Preprint,
arXiv:2205.13147.
Sean Lee, Aamir Shakir, Darius Koenig, and Julius
Lipp. 2024. Open source strikes bread - new fluffy
embeddings model.
Patrick Lewis, Ethan Perez, Aleksandara Piktus, Fabio
Petroni, Vladimir Karpukhin, Naman Goyal, Hein-
rich Kuttler, Mike Lewis, Wen tau Yih, Tim Rock-
täschel, Sebastian Riedel, and Douwe Kiela. 2020.
Retrieval-augmented generation for knowledge-
intensive nlp tasks. ArXiv, abs/2005.11401.
Kyle Lo, Lucy Lu Wang, Mark Neumann, Rod-
ney Michael Kinney, and Daniel S. Weld. 2020.
S2orc: The semantic scholar open research corpus.
In Annual Meeting of the Association for Computa-
tional Linguistics.
Niklas Muennighoff, Nouamane Tazi, Loic Magne, and
Nils Reimers. 2022. Mteb: Massive text embedding
benchmark. In Conference of the European Chapter
of the Association for Computational Linguistics.
Benjamin Newman, Yoonjoo Lee, Aakanksha Naik,
Pao Siangliulue, Raymond Fok, Juho Kim, Daniel S.
Weld, Joseph Chee Chang, and Kyle Lo. 2024. Arx-
ivdigestables: Synthesizing scientific literature into
tables using language models. In Conference on Em-
pirical Methods in Natural Language Processing.
Jakob Nielsen. 1994. Enhancing the explanatory power
of usability heuristics. In Proceedings of the SIGCHI
conference on Human Factors in Computing Systems,
pages 152–158.
OpenAI. 2024a. Chatgpt – deep research mode. Ac-
cessed: 2025-03-28.
OpenAI. 2024b. Openai o1 system card.
Perplexity. 2024. Perplexity ai – ask anything. Ac-
cessed: 2025-03-28.
Nils Reimers and Iryna Gurevych. 2019. Sentence-bert:
Sentence embeddings using siamese bert-networks.
In Proceedings of the 2019 Conference on Empirical
Methods in Natural Language Processing. Associa-
tion for Computational Linguistics.
Scite. 2024. Scite – smart citations for research. Ac-
cessed: 2025-03-28.
Aamir Shakir, Darius Koenig, Julius Lipp, and Sean Lee.
2024. Boost your search with the crispy mixedbread
rerank models.
Yijia Shao, Yucheng Jiang, Theodore Kanell, Peter Xu,
Omar Khattab, and Monica Lam. 2024a. Assisting
in writing Wikipedia-like articles from scratch with
large language models. In Proceedings of the 2024
Conference of the North American Chapter of the
Association for Computational Linguistics: Human
Language Technologies (Volume 1: Long Papers),
pages 6252–6278, Mexico City, Mexico. Association
for Computational Linguistics.
Yijia Shao, Yucheng Jiang, Theodore A. Kanell, Pe-
ter Xu, Omar Khattab, and Monica S. Lam. 2024b.
Assisting in writing wikipedia-like articles from
scratch with large language models. Preprint,
arXiv:2402.14207.
Michael D. Skarlinski, Sam Cox, Jon M. Laurent,
James D. Braza, Michaela M. Hinks, Michael J
Hammerling, Manvitha Ponnapati, Samuel G. Ro-
driques, and Andrew D. White. 2024. Language
agents achieve superhuman synthesis of scientific
knowledge. ArXiv, abs/2409.13740.
Aviv Slobodkin, Eran Hirsch, Arie Cattan, Tal Schuster,
and Ido Dagan. 2024. Attribute first, then generate:
Locally-attributable grounded text generation. In
Annual Meeting of the Association for Computational
Linguistics.
Aivin V. Solatorio. 2024. Gistembed: Guided in-sample
selection of training negatives for text embedding
fine-tuning. ArXiv, abs/2402.16829.
Saba Sturua, Isabelle Mohr, Mohammad Kalim Akram,
Michael Günther, Bo Wang, Markus Krimmel, Feng
Wang, Georgios Mastrapas, Andreas Koukounas, An-
dreas Koukounas, Nan Wang, and Han Xiao. 2024.
jina-embeddings-v3: Multilingual embeddings with
task lora. Preprint, arXiv:2409.10173.
Liang Wang, Nan Yang, Xiaolong Huang, Binxing
Jiao, Linjun Yang, Daxin Jiang, Rangan Majumder,
and Furu Wei. 2022. Text embeddings by weakly-
supervised contrastive pre-training. arXiv preprint
arXiv:2212.03533.
Xiangchao Yan, Shiyang Feng, Jiakang Yuan, Renqiu
Xia, Bin Wang, Bo Zhang, and Lei Bai. 2025. Sur-
veyforge: On the outline heuristics, memory-driven
generation, and multi-dimensional evaluation for au-
tomated survey writing.
You.com. 2024. You.com – personalized ai search. Ac-
cessed: 2025-03-28.
Brian Zhang, Eric Mitchell, Hongyu Ren, Kevin Lu,
Max Schwarzer, Michelle Pokrass, Shengjia Zhao,
Ted Sanders, Adam Kalai, Alexandre Passos, Ben-
jamin Sokolowsky, Elaine Ya Le, Erik Ritter, Hao
Sheng, Hanson Wang, Ilya Kostrikov, James Lee, Jo-
hannes Ferstad, Michael Lampe, and 93 others. 2025.
Openai o3-mini system card.
Zekun Zhou, Xiaocheng Feng, Lei Huang, Xiachong
Feng, Ziyun Song, Ruihan Chen, Liang Zhao, Weitao
Ma, Yuxuan Gu, Baoxin Wang, Dayong Wu, Guop-
ing Hu, Ting Liu, and Bing Qin. 2025. From hy-
pothesis to publication: A comprehensive survey of
ai-driven research support systems.
A Python Package Usage
Figure 5 shows a minimal example of running
the system pipeline with the ai2-scholar-qa python
package and how every component can be extended
or modified as the users see fit.
HuggingFace embedding model name
Snowflake/snowflake-arctic-embed-m5
sentence-transformers/all-mpnet-base-v2
(Reimers and Gurevych, 2019)
avsolatorio/GIST-Embedding-v0 (Solatorio, 2024)
Snowflake/snowflake-arctic-embed-m-long 6
intfloat/e5-base-v2 (Wang et al., 2022)
mixedbread-ai/mxbai-embed-large-v1
(Lee et al., 2024)
jinaai/jina-embeddings-v3 (Sturua et al., 2024)
Table 4: Embedding Models to optimize retrieval
E Retrieval Ensemble Experiments
Figure 6 shows results of our ensembling experi-
ments for the full-text retrieval index. SparseEm-
bed introduces an overhead with minimal perfor-
mance gains, so we picked an ensemble of em-
bedding similarity and BM25 as our final ranking
metric.
Figure 5: ai2-scholar-qa usage example
B Document Relevance Prompt
We used the following prompt to obtain binary rele-
vance labels, which agreed with human annotators
80% of the time:
If any part of the following text
is relevant to the following question,
then return 1, otherwise return 0.
Non-english results are not relevant,
results which are primarily tables are
not relevant.
C Retrieval Tuning Query Generation
Queries for the dev set were obtained from three
internal sources of human research questions, and a
set of LLM generations. We experimented with sev-
eral methods for constructing the synthetic LLM
questions. Our approach was to generate questions
similar to those asked by real users by prompting
the LLM to output: (1) a question based on para-
graphs retrieved from the corpus, and (2) a "more
general" version of the first question. We only use
the "more general" set since they were more similar
to real user queries.
D Embedding Models for Retrieval
We experimented with multiple top embedding
models from the MTEB leader board to optimize
retrieval for our system. These are outlined in Ta-
ble 4.
Figure 6: Ranking performance for various ensembles
with relative size of the index required. Excluding
SparseEmbed reduces the index size by 20% without a
significant drop in ranking performance.
F Prompt for Evaluating Attribution
As an Attribution Validator, your task
is to verify whether a given reference
can support the given claim. A claim can
be either a plain sentence or a question
followed by its answer. Specifically,
your response should clearly indicate
the relationship: Attributable,
Contradictory or Extrapolatory. A
contradictory error occurs when you
can infer that the answer contradicts
the fact presented in the context,
while an extrapolatory error means that
you cannot infer the correctness of
the answer based on the information
provided in the context. Output your
response as a json with only a single
key "output" and a value of one among
- ("Attributable", "Contradictory",
"Extrapolatory").
Claim: claim
Reference: ref_excerpt
G User Feedback Examples
Table 5 lists some examples of the user complaints
for Scholar QA reports.
Feedback
The structure is good, but the articles you choose are not
from top journals.
The first citation says that rabbits can obtain cholesterol
from diet, not rats.
These provide a lot of general information about the topic,
but nothing here actually addresses the central question I
asked.
The answer did not address the ‘MOBILIZATION’ tech-
niques at all! The answer is wrong because it addressed
Exercise therapy!
They address the general setting, but not the specific question
I asked.
It’s only analysing on SASAF model, but there are more.
Table 5: Example Feedback on Research Issues
H Progress Updates and Report Sections
Figure 7 demonstrates how we display in real-time
the progress of the system during generation. This
included number of papers and passages the were
processed in each step, as well as the outline as it
is being generated. Each section appears as soon
as it is generated, so users can begin browsing the
first sections.
Figure 7: Progress indication and section streaming.
I Query Type Analysis
To analyze the types of questions users are asking,
we use an LLM to categorize the queries. The most
Figure 8: Distribution of different question types sub-
mitted to Scholar QA deployed Web application.
prominent types were comprehensive deep-dive
into a specific research topic (15k) and comparative
analysis of prior work (5k). Other themes such as
factoid QA or specific methods, datasets accounted
for fewer queries.
J Generation Results with updated
GPT-4o
Table 6 shows results on ScholarQA-CS with
gpt-4o-2024-11-20 as the LLM judge. These
results can be contrasted with the first two
columns in Table 2 which are obtained with
gpt-4o-2024-08-06 as the judge. Even though
the absolute scores are inflated compared to Ta-
ble 2, the relative rankings are about the same with
Scholar QA getting the best overall score.
Model Score Model Score
Rubrics Total Rubrics Total
LLM Prompting (No Retrieval) QA Systems
Llama 3.1-8B 51.8 48.2 Llama 3.1-70B 57.0 51.2 Claude 3.5 S 57.8 51.3 Claude 3.7 S 68.4 60.8 +Thinking 68.3 58.7 GPT-4.1 69.3 61.8 SQA-Claude 3.7 S 67.3 67.2
SQA-Claude 3.5 S 61.3 67.1
OS-GPT-4o 54.9 59.9
PaperQA2 43.8 54.1
Perplex. Sonar DR 43.9 56.0
STORM 59.2 64.7
o1-mini 69.1 61.3
o3-mini 68.5 55.9
Table 6: Evaluation results on ScholarQA-CS bench-
mark with gpt-4o-2024-11-20 as the judge. System
responses are either generated by simply prompting
LLMs with the questions or by issuing the queries to
RAG based QA systems. Expert annotated rubrics
only scores are reported in addition to the overall to-
tal. The overall best results are highlighted and best
results within a category are underlined. SQA: Ai2
Scholar QA, OS: Open Scholar, S: Sonnet, Claude 3.5
S: claude-3-5-sonnet-20241022.""",
    },
    {
        "id": "paper2",
        "title": "SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROUGH SELF-REFLECTION",
        "abstract": """Despite their remarkable capabilities, large language models (LLMs) often produce
responses containing factual inaccuracies due to their sole reliance on the paramet-
ric knowledge they encapsulate. Retrieval-Augmented Generation (RAG), an ad
hoc approach that augments LMs with retrieval of relevant knowledge, decreases
such issues. However, indiscriminately retrieving and incorporating a fixed number
of retrieved passages, regardless of whether retrieval is necessary, or passages are
relevant, diminishes LM versatility or can lead to unhelpful response generation.
We introduce a new framework called Self-Reflective Retrieval-Augmented Gen-
eration (SELF-RAG) that enhances an LM’s quality and factuality through retrieval
and self-reflection. Our framework trains a single arbitrary LM that adaptively
retrieves passages on-demand, and generates and reflects on retrieved passages
and its own generations using special tokens, called reflection tokens. Generating
reflection tokens makes the LM controllable during the inference phase, enabling it
to tailor its behavior to diverse task requirements. Experiments show that SELF-
RAG (7B and 13B parameters) significantly outperforms state-of-the-art LLMs
and retrieval-augmented models on a diverse set of tasks. Specifically, SELF-RAG
outperforms ChatGPT and retrieval-augmented Llama2-chat on Open-domain QA,
reasoning and fact verification tasks, and it shows significant gains in improving
factuality and citation accuracy for long-form generations relative to these models.1""",
        "text": """arXiv:2310.11511v1 [cs.CL] 17 Oct 2023
Preprint.
SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND
CRITIQUE THROUGH SELF-REFLECTION
Akari Asai†, Zeqiu Wu†, Yizhong Wang†§, Avirup Sil‡, Hannaneh Hajishirzi†§
†University of Washington §Allen Institute for AI ‡IBM Research AI
{akari,zeqiuwu,yizhongw,hannaneh}@cs.washington.edu, avi@us.ibm.com
ABSTRACT
Despite their remarkable capabilities, large language models (LLMs) often produce
responses containing factual inaccuracies due to their sole reliance on the paramet-
ric knowledge they encapsulate. Retrieval-Augmented Generation (RAG), an ad
hoc approach that augments LMs with retrieval of relevant knowledge, decreases
such issues. However, indiscriminately retrieving and incorporating a fixed number
of retrieved passages, regardless of whether retrieval is necessary, or passages are
relevant, diminishes LM versatility or can lead to unhelpful response generation.
We introduce a new framework called Self-Reflective Retrieval-Augmented Gen-
eration (SELF-RAG) that enhances an LM’s quality and factuality through retrieval
and self-reflection. Our framework trains a single arbitrary LM that adaptively
retrieves passages on-demand, and generates and reflects on retrieved passages
and its own generations using special tokens, called reflection tokens. Generating
reflection tokens makes the LM controllable during the inference phase, enabling it
to tailor its behavior to diverse task requirements. Experiments show that SELF-
RAG (7B and 13B parameters) significantly outperforms state-of-the-art LLMs
and retrieval-augmented models on a diverse set of tasks. Specifically, SELF-RAG
outperforms ChatGPT and retrieval-augmented Llama2-chat on Open-domain QA,
reasoning and fact verification tasks, and it shows significant gains in improving
factuality and citation accuracy for long-form generations relative to these models.1
1 INTRODUCTION
State-of-the-art LLMs continue to struggle with factual errors (Mallen et al., 2023; Min et al., 2023)
despite their increased model and data scale (Ouyang et al., 2022). Retrieval-Augmented Generation
(RAG) methods (Figure 1 left; Lewis et al. 2020; Guu et al. 2020) augment the input of LLMs
with relevant retrieved passages, reducing factual errors in knowledge-intensive tasks (Ram et al.,
2023; Asai et al., 2023a). However, these methods may hinder the versatility of LLMs or introduce
unnecessary or off-topic passages that lead to low-quality generations (Shi et al., 2023) since they
retrieve passages indiscriminately regardless of whether the factual grounding is helpful. Moreover,
the output is not guaranteed to be consistent with retrieved relevant passages (Gao et al., 2023) since
the models are not explicitly trained to leverage and follow facts from provided passages. This
work introduces Self-Reflective Retrieval-augmented Generation (SELF-RAG) to improve an
LLM’s generation quality, including its factual accuracy without hurting its versatility, via on-demand
retrieval and self-reflection. We train an arbitrary LM in an end-to-end manner to learn to reflect on
its own generation process given a task input by generating both task output and intermittent special
tokens (i.e., reflection tokens). Reflection tokens are categorized into retrieval and critique tokens to
indicate the need for retrieval and its generation quality respectively (Figure 1 right). In particular,
given an input prompt and preceding generations, SELF-RAG first determines if augmenting the
continued generation with retrieved passages would be helpful. If so, it outputs a retrieval token that
calls a retriever model on demand (Step 1). Subsequently, SELF-RAG concurrently processes multiple
retrieved passages, evaluating their relevance and then generating corresponding task outputs (Step
2). It then generates critique tokens to criticize its own output and choose best one (Step 3) in terms
of factuality and overall quality. This process differs from conventional RAG (Figure 1 left), which
1Our code and trained models are available at https://selfrag.github.io/.
1
Preprint.
Retrieval-Augmented Generation (RAG) Prompt How did US states get their names?
Step 1: Retrieve K documents
Of the fifty states, eleven are named
after an individual person.
Popular names by states. In Texas,
Emma is a popular baby name.
Retriever
California was named after a fictional
island in a Spanish book.
Step 2: Prompt LM with K docs and generate
Prompt How did US states get their names? +
LM
US states got their names from a variety of
sources. Eleven states are named after an
individual person (e.g, California was named
after Christopher Columbus). Some states
including Texas and Utah, are named after
Contradictory
Native American tribe.
No information in passages
Prompt + Prompt +
Ours: Self-reflective Retrieval-Augmented Generation (Self-RAG)
Prompt How did US states get their names?
Step 1: Retrieve on demand
US states got their names from a variety of sources. Retrieve
Step 2: Generate segment in parallel
Prompt +
11 of 50 state names
Texas is named
come from persons. after a Native American tribe.
California's name has its
origins in a 16th-century novel
Las Sergas de Esplandián.
Step 3: Critique outputs and select best segment
>
>
Retrieve
US states got their names from a variety of sources. 11 of 50
states names are come from persons. 26 states are named
after Native Americans, including Utah.
Relevant
Irrelevant Supported
Relevant
Partially
Repeat.…
Prompt: Write an essay of your best summer vacation Prompt: Write an essay of your best summer vacation
My best summer vacation is when my family and I embarked on a road trip along …
No Retrieval My best…
Figure 1: Overview of SELF-RAG. SELF-RAG learns to retrieve, critique, and generate text passages
to enhance overall generation quality, factuality, and verifiability.
consistently retrieves a fixed number of documents for generation regardless of the retrieval necessity
(e.g., the bottom figure example does not require factual knowledge) and never second visits the
generation quality. Moreover, SELF-RAG provides citations for each segment with its self-assessment
of whether the output is supported by the passage, leading to easier fact verification.
SELF-RAG trains an arbitrary LM to generate text with reflection tokens by unifying them as the
next token prediction from the expanded model vocabulary. We train our generator LM on a diverse
collection of text interleaved with reflection tokens and retrieved passages. Reflection tokens, inspired
by reward models used in reinforcement learning (Ziegler et al., 2019; Ouyang et al., 2022), are
inserted offline into the original corpus by a trained critic model. This eliminates the need to host a
critic model during training, reducing overhead. The critic model, in part, is supervised on a dataset
of input, output, and corresponding reflection tokens collected by prompting a propriety LM (i.e.,
GPT-4; OpenAI 2023). While we draw inspiration from studies that use control tokens to start and
guide text generation (Lu et al., 2022; Keskar et al., 2019), our trained LM uses critique tokens to
assess its own predictions after each generated segment as an integral part of the generation output.
SELF-RAG further enables a customizable decoding algorithm to satisfy hard or soft constraints,
which are defined by reflection token predictions. In particular, our inference-time algorithm enables
us to (1) flexibly adjust retrieval frequency for different downstream applications and (2) customize
models’ behaviors to user preferences by leveraging reflection tokens through segment-level beam
search using the weighted linear sum of the reflection token probabilities as segment score.
Empirical results on six tasks, including reasoning and long-form generation, demonstrate that SELF-
RAG significantly outperforms pre-trained and instruction-tuned LLMs that have more parameters and
widely adopted RAG approaches with higher citation accuracy. In particular, SELF-RAG outperforms
retrieval-augmented ChatGPT on four tasks, Llama2-chat (Touvron et al., 2023) and Alpaca (Dubois
et al., 2023) on all tasks. Our analysis demonstrates the effectiveness of training and inference with
reflection tokens for overall performance improvements as well as test-time model customizations
(e.g., balancing the trade-off between citation previsions and completeness).
2 RELATED WORK
Retrieval-Augmented Generation. Retrieval-Augmented Generation (RAG) augments the input
space of LMs with retrieved text passages (Guu et al., 2020; Lewis et al., 2020), leading to large
improvements in knowledge-intensive tasks after fine-tuning or used with off-the-shelf LMs (Ram
et al., 2023). A more recent work (Luo et al., 2023) instruction-tunes an LM with a fixed number
2
Preprint.
of retrieved passages prepended to input, or pre-train a retriever and LM jointly, followed by few-
shot fine-tuning on task datasets (Izacard et al., 2022b). While prior work often retrieves only
once at the beginning, Jiang et al. (2023) propose to adaptively retrieve passages for generation
on top of a proprietary LLM or Schick et al. (2023) train an LM to generate API calls for named
entities. Yet, the improved task performance of such approaches often comes at the expense of
runtime efficiency (Mallen et al., 2023), robustness to irrelevant context (Shi et al., 2023), and lack of
attributions (Liu et al., 2023a; Gao et al., 2023). We introduce a method to train an arbitrary LM to
learn to use retrieval on-demand for diverse instruction-following queries and introduce controlled
generation guided by reflections tokens to further improve generation quality and attributions.
Concurrent RAG work. A few concurrent works2 on RAG propose new training or prompting
strategies to improve widely-adopted RAG approaches. Lin et al. (2023) fine-tune both the retriever
and LM on instruction-tuning datasets in two steps. While we also train our model on diverse
instruction-following datasets, SELF-RAG enables retrieval on demand and selection of the best
possible model output via fine-grained self-reflection, making it widely applicable and more robust
and controllable. Yoran et al. (2023) use a natural language inference model and Xu et al. (2023) use
a summarization model to filter out or compress retrieved passages before using them to prompt the
LM to generate the output. SELF-RAG processes passages in parallel and filters out irrelevant ones
through self-reflection, without relying on external models at inference. Moreover, our self-reflection
mechanism also evaluates other aspects of the model output quality including factuality. LATS (Zhou
et al., 2023) prompt off-the-shelf LMs to search for relevant information for question answering tasks
and to generate with tree search, guided by LM-generated value scores. While their value function
simply indicates an overall score of each generation, SELF-RAG trains to an arbitrary LM to learn to
generate fine-grained self-reflection and customizable inference.
Training and generating with critics. Training LLMs with reinforcement learning (e.g., Proximal
Policy Optimization or PPO; Schulman et al. 2017) from human feedback (RLHF) has proven
effective in aligning LLMs with human preferences (Ouyang et al., 2022). Wu et al. (2023) introduce
fine-grained RLHF with multiple reward models. Though our work also studies fine-grained critique
on retrieval and generation, we train our target LM on task examples augmented with reflection
tokens from a critic model offline, with a far lower training cost compared to RLHF. In addition,
reflection tokens in SELF-RAG enable controllable generation at inference, while RLHF focuses on
human preference alignment during training. Other works use general control tokens to guide LM
generation (Lu et al., 2022; Korbak et al., 2023), while SELF-RAG uses reflection tokens to decide the
need for retrieval and to self-evaluate generation quality. Xie et al. (2023) propose a self-evaluation-
guided decoding framework, but they focus only on reasoning tasks with one evaluation dimension
(reasoning path consistency) and without retrieval. Recent work on LLM refinement (Dhuliawala
et al., 2023; Madaan et al., 2023; Paul et al., 2023) prompts a model to generate task output, natural
language feedback and refined task output iteratively, but at the cost of inference efficiency.
3 SELF-RAG: LEARNING TO RETRIEVE, GENERATE AND CRITIQUE
We introduce Self-Reflective Retrieval-Augmented Generation (SELF-RAG), shown in Figure 1.
SELF-RAG is a framework that enhances the quality and factuality of an LLM through retrieval and
self-reflection, without sacrificing LLM’s original creativity and versatility. Our end-to-end training
lets an LM Mgenerate text informed by retrieved passages, if needed, and criticize the output by
learning to generate special tokens. These reflection tokens (Table 1) signal the need for retrieval
or confirm the output’s relevance, support, or completeness. In contrast, common RAG approaches
retrieve passages indiscriminately, without ensuring complete support from cited sources.
3.1 PROBLEM FORMALIZATION AND OVERVIEW
Formally, given input x, we train Mto sequentially generate textual outputs yconsisting of multiple
segments y= [y1,...,yT ], where yt indicates a sequence of tokens for the t-th segment.3 Generated
tokens in yt include text from the original vocabulary as well as the reflection tokens (Table 1).
2All work is arXived within a week of this preprint.
3In this paper, we treat one sentence as a segment in our experiments, but our framework is applicable to any
segment unit (i.e., sub-sentence).
3
Preprint.
Type Input Output Definitions
Retrieve x / x, y {yes, no, continue} Decides when to retrieve with R
ISREL x, d {relevant, irrelevant} d provides useful information to solve x.
ISSUP x, d, y {fully supported, partially
All of the verification-worthy statement in y
supported, no support}
is supported by d.
x, y {5, 4, 3, 2, 1} y is a useful response to x.
ISUSE Table 1: Four types of reflection tokens used in SELF-RAG. Each type uses several tokens to represent
its output values. The bottom three rows are three types of Critique tokens, and the bold text indicates
the most desirable critique tokens. x,y,dindicate input, output, and a relevant passage, respectively.
Algorithm 1 SELF-RAG Inference
Require: Generator LM M, Retriever R, Large-scale passage collections {d1,...,dN }
1: Input: input prompt xand preceding generation y<t, Output: next output segment yt
2: Mpredicts Retrieve given (x,y<t)
3: if Retrieve == Yes then
4: Retrieve relevant text passages D using Rgiven (x,yt−1) 5: Mpredicts ISREL given x,dand yt given x,d,y<t for each d∈D 6: Mpredicts ISSUP and ISUSE given x,yt,dfor each d∈D 7: Rank yt based on ISREL , ISSUP , ISUSE 8: else if Retrieve == No then
9: Mgen predicts yt given x 10: Mgen predicts ISUSE given x,yt ▷Retrieve
▷Generate
▷Critique
▷Detailed in Section 3.3
▷Generate
▷Critique
Inference overview. Figure 1 and Algorithm 1 present an overview of SELF-RAG at inference. For
every xand preceding generation y<t, the model decodes a retrieval token to evaluate the utility
of retrieval. If retrieval is not required, the model predicts the next output segment, as it does in a
standard LM. If retrieval is needed, the model generates: a critique token to evaluate the retrieved
passage’s relevance, the next response segment, and a critique token to evaluate if the information in
the response segment is supported by the passage. Finally, a new critique token evaluates the overall
utility of the response.4 To generate each segment, SELF-RAG processes multiple passages in parallel
and uses its own generated reflection tokens to enforce soft constraints (Section 3.3) or hard control
(Algorithm 1) over the generated task output. For instance, in Figure 1 (right), the retrieved passages
d1 is selected at the first time step since d2 does not provide direct evidence ( ISREL is Irrelevant)
and d3 output is only partially supported while d1 are fully supported.
Training overview. SELF-RAG enables an arbitrary LM to generate text with reflection tokens
by unifying them as next token predictions from the expanded model vocabulary (i.e., the original
vocabulary plus reflection tokens). Specifically, we train the generator model Mon a curated corpus
with interleaving passages retrieved by a retriever Rand reflection tokens predicted by a critic model
C(summarized in Appendix Algorithm 2). We train Cto generate reflection tokens for evaluating
retrieved passages and the quality of a given task output (Section 3.2.1). Using the critic model, we
update the training corpus by inserting reflection tokens into task outputs offline. Subsequently, we
train the final generator model (M) using the conventional LM objective (Section 3.2.2) to enable
Mto generate reflection tokens by itself without relying on the critic at inference time.
3.2 SELF-RAG TRAINING
Here, we describe the supervised data collection and training of two models, the critic C(Section 3.2.1)
and the generator M(Section 3.2.2).
3.2.1 TRAINING THE CRITIC MODEL
Data collection for critic model. Manual annotation of reflection tokens for each segment is
expensive (Wu et al., 2023). A state-of-the-art LLM like GPT-4 (OpenAI, 2023) can be effectively
4We follow Liu et al. (2023a) in using a “perceived” utility value that is independent of retrieved passages.
4
Preprint.
Input: Write an essay of your best summer vacation
Output: My best summer vacation was a magical escape
to the coastal town of Santorini. The azure waters,
charming white-washed building are unforgettable.
Input: How did US states get their names?
Output: 1 of 50 states names come from persons. For instance, Louisiana was named in honor
of King Louis XIV of France and Georgia was named after King George II.
Retriever
Critic LM
Augmented Output: Retrieve
<p>Of the fifty states, eleven are named after an individual person</p>.
Augmented Output: My best summer
No Retrieval
vacation was a magical escape to the coastal town of
11 of 50 states’ names come from person. Relevant Santorini. The azure waters, charming white-
No Retrieval
washed building are unforgettable experience.
Util: 5
honor of Louis XIV of France.</p>. Retrieve
Supported
<p>LOUISIANA: Named in
For instance, Louisiana was named after King Louis XIV, and
Georgia was named after King George II.
Partially
Relevant Util: 5
Figure 2: SELF-RAG training examples. The left example does not require retrieval while the right
one requires retrieval; thus, passages are inserted. More examples are in Appendix Table 4.
(1)
used to generate such feedback (Liu et al., 2023b). However, depending on such proprietary LMs
can raise API costs and diminish reproducibility (Chen et al., 2023). We create supervised data by
prompting GPT-4 to generate reflection tokens and then distill their knowledge into an in-house C.
For each group of reflection tokens, we randomly sample instances from the original training data:
{Xsample,Ysample}∼{X,Y}. As different reflection token groups have their own definitions and
input, as shown in Table 1, we use different instruction prompts for them. Here, we use Retrieve as
an example. We prompt GPT-4 with a type-specific instruction (“Given an instruction, make a
judgment on whether finding some external documents from the web helps to generate a better
response.”) followed by few-shot demonstrations I the original task input xand output yto predict
an appropriate reflection token as text: p(r|I,x,y). Manual assessment reveals that GPT-4 reflection
token predictions show high agreement with human evaluations. We collect 4k-20k supervised
training data for each type and combine them to form training data for C. Appendix Section D shows
the full list of instructions, and A.1 contains more details and our analysis.
Critic learning. After we collect training data Dcritic, we initialize Cwith a pre-trained LM and
train it on Dcritic using a standard conditional language modeling objective, maximizing likelihood:
max
E((x,y),r)∼Dcritic log pC(r|x,y), rfor reflection tokens. C
Though the initial model can be any pre-trained LM, we use the same one as the generator LM
(i.e., Llama 2-7B; Touvron et al. 2023) for Cinitialization. The critic achieves a higher than 90%
agreement with GPT-4-based predictions on most reflection token categories (Appendix Table 5).
3.2.2 TRAINING THE GENERATOR MODEL
Data collection for generator. Given an input-output pair (x,y), we augment the original output
y using the retrieval and critic models to create supervised data that precisely mimics the SELF-
RAG inference-time process (Section 3.1). For each segment yt ∈y, we run Cto assess whether
additional passages could help to enhance generation. If retrieval is required, the retrieval special
token Retrieve =Yes is added, and Rretrieves the top K passages, D. For each passage, Cfurther
evaluates whether the passage is relevant and predicts ISREL . If a passage is relevant, Cfurther
evaluates whether the passage supports the model generation and predicts ISSUP . Critique tokens
ISREL and ISSUP are appended after the retrieved passage or generations. At the end of the output, y
(or yT ), Cpredicts the overall utility token ISUSE , and an augmented output with reflection tokens
and the original input pair is added to Dgen. See the example training data in Figure 2.
Generator learning. We train the generator model Mby training on the curated corpus augmented
with reflection tokens Dgen using the standard next token objective:
max
E(x,y,r)∼Dgen log pM(y,r|x). (2)
M
Unlike Ctraining (Eq. 1), Mlearns to predict the target output as well as the reflection tokens. During
training, we mask out the retrieved text chunks (surrounded by <p> and </p> in Figure 2) for loss
calculation and expand the original vocabulary Vwith a set of reflection tokens {Critique , Retrieve }.
Connections to prior work on learning with critique. Recent work incorporates additional
critique (feedback) during training, e.g., RLHF (Ouyang et al. 2022) via PPO. While PPO relies on
5
Preprint.
separate reward models during training, we compute critique offline and directly insert them into the
training corpus, where the generator LM is trained with a standard LM objective. This significantly
reduces training costs compared to PPO. Our work also relates to prior work that incorporates special
tokens to control generation (Keskar et al., 2019; Lu et al., 2022; Korbak et al., 2023). Our SELF-RAG
learns to generate special tokens to evaluate its own prediction after each generated segment, enabling
the use of a soft re-ranking mechanism or hard constraints at inference (discussed next).
3.3 SELF-RAG INFERENCE
Generating reflection tokens to self-evaluate its own output makes SELF-RAG controllable during the
inference phase, enabling it to tailor its behavior to diverse task requirements. For tasks demanding
factual accuracy (Min et al., 2023), we aim for the model to retrieve passages more frequently to
ensure that the output aligns closely with the available evidence. Conversely, in more open-ended
tasks, like composing a personal experience essay, the emphasis shifts towards retrieving less and
prioritizing the overall creativity or utility score. In this section, we describe approaches to enforce
control to meet these distinct objectives during the inference process.
Adaptive retrieval with threshold. SELF-RAG dynamically decides when to retrieve text passages by
predicting Retrieve . Alternatively, our framework allows a threshold to be set. Specifically, if the prob-
ability of generating the Retrieve =Yes token normalized over all output tokens in Retrieve surpasses a
designated threshold, we trigger retrieval (details in Appendix Section A.3).
Tree-decoding with critique tokens. At each segment step t, when retrieval is required, based either
on hard or soft conditions, Rretrieves Kpassages, and the generator Mprocesses each passage in
parallel and outputs Kdifferent continuation candidates. We conduct a segment-level beam search
(with the beam size=B) to obtain the top-Bsegment continuations at each timestamp t, and return
the best sequence at the end of generation. The score of each segment yt with respect to passage dis
updated with a critic score Sthat is the linear weighted sum of the normalized probability of each
Critique token type. For each critique token group G(e.g., ISREL ), we denote its score at timestamp
tas sG
t , and we compute a segment score as follows:
f(yt,d, Critique ) = p(yt|x,d,y<t)) + S( Critique ),where wGsG
t for G= {ISREL , ISSUP , ISUSE }, (3)
(4)
S( Critique ) =
G∈G
where sG
t = pt (ˆ r)
N G
i=1 pt (ri ) stands for the generation probability of the most desirable reflection token
ˆ
r(e.g., ISREL =Relevant) for the critique token type Gwith NG distinct tokens (that represent
different possible values for G). The weights wG in Eq. 4 are hyperparameters that can be adjusted
at inference time to enable customized behaviors at test time. For instance, to ensure that result
y is mostly supported by evidence, we can set a weight term for the ISSUP score higher, while
relatively lowering weights for other aspects. Alternatively, we could further enforce hard constraints
during decoding using Critique . Instead of using a soft reward function in Eq. 4, we could explicitly
filter out a segment continuation when the model generates an undesirable Critique token (e.g.,
ISSUP =No support) . Balancing the trade-off between multiple preferences has been studied
in RLHF (Touvron et al., 2023; Wu et al., 2023), which often requires training to change models’
behaviors. SELF-RAG tailors an LM with no additional training.
4 EXPERIMENTS
4.1 TASKS AND DATASETS
We conduct evaluations of our SELF-RAG and diverse baselines on a range of downstream tasks,
holistically evaluating outputs with metrics designed to assess overall correctness, factuality, and
fluency. Throughout these experiments, we conduct zero-shot evaluations, where we provide instruc-
tions describing tasks without few-shot demonstrations (Wei et al., 2022; Sanh et al., 2022). Details of
our experiments’ settings, including test-time instructions, are available in the Appendix Section B.1.
Closed-set tasks include two datasets, i.e., a fact verification dataset about public health (PubHealth;
Zhang et al. 2023) and a multiple-choice reasoning dataset created from scientific exams (ARC-
6
Preprint.
Challenge; Clark et al. 2018). We use accuracy as an evaluation metric and report on the test set. We
aggregate the answer probabilities of target classes for both of these datasets (Appendix Section B.2).
Short-form generations tasks include two open-domain question answering (QA) datasets,
PopQA (Mallen et al., 2023) and TriviaQA-unfiltered (Joshi et al., 2017), where systems need
to answer arbitrary questions about factual knowledge. For PopQA, we use the long-tail subset,
consisting of 1,399 rare entity queries whose monthly Wikipedia page views are less than 100. As the
TriviaQA-unfiltered (open) test set is not publicly available, we follow prior work’s validation and
test split (Min et al., 2019; Guu et al., 2020), using 11,313 test queries for evaluation. We evaluate
performance based on whether gold answers are included in the model generations instead of strictly
requiring exact matching, following Mallen et al. (2023); Schick et al. (2023).
Long-form generation tasks include a biography generation task (Min et al., 2023) and a long-form
QA task ALCE-ASQA Gao et al. (2023); Stelmakh et al. (2022). We use FactScore (Min et al.,
2023) to evaluate biographies, and we use official metrics of correctness (str-em), fluency based on
MAUVE (Pillutla et al., 2021), and citation precision and recall (Gao et al., 2023) for ASQA. 5
4.2 BASELINES
Baselines without retrievals. We evaluate strong publicly available pre-trained LLMs,
Llama27B,13B (Touvron et al., 2023), instruction-tuned models, Alpaca7B,13B (Dubois et al., 2023)
(our replication based on Llama2); and models trained and reinforced using private data, Chat-
GPT (Ouyang et al., 2022) and Llama2-chat13B. For instruction-tuned LMs, we use the official
system prompt or instruction format used during training if publicly available. We also compare our
method to concurrent work, CoVE65B (Dhuliawala et al., 2023), which introduces iterative prompt
engineering to improve the factuality of LLM generations.
Baselines with retrievals. We evaluate models augmented with retrieval at test time or during training.
The first category includes standard RAG baselines, where an LM (Llama2, Alpaca) generates output
given the query prepended with the top retrieved documents using the same retriever as in our system.
It also includes Llama2-FT, where Llama2 is fine-tuned on all training data we use without the
reflection tokens or retrieved passages. We also report the result of retrieval-augmented baselines
with LMs trained with private data: Ret-ChatGPT and Ret-Llama2-chat, which deploy the same
augmentation technique above, as well as perplexity.ai, an InstructGPT-based production search
system. The second category includes concurrent methods that are trained with retrieved text
passages, i.e., SAIL (Luo et al., 2023) to instruction-tune an LM on the Alpaca instruction-tuning
data with top retrieved documents inserted before instructions, and Toolformer (Schick et al., 2023)
to pre-train an LM with API calls (e.g., Wikipedia APIs).6
4.3 EXPERIMENTAL SETTINGS
Training data and settings. Our training data consists of diverse instruction-following input-output
pairs. In particular, we sample instances from Open-Instruct processed data (Wang et al., 2023) and
knowledge-intensive datasets (Petroni et al., 2021; Stelmakh et al., 2022; Mihaylov et al., 2018). In
total, we use 150k instruction-output pairs. We use Llama2 7B and 13B (Touvron et al., 2023) as
our generator base LM, and we use Llama2 7B as our base critic LM. For the retriever model R, we
use off-the-shelf Contriever-MS MARCO (Izacard et al., 2022a) by default and retrieve up to ten
documents for each input. More training details are in the Appendix Section B.1.
Inference settings. As a default configuration, we assign the weight terms ISREL , ISSUP , ISUSE
values of 1.0, 1.0 and 0.5, respectively. To encourage frequent retrieval, we set the retrieval threshold
to 0.2 for most tasks and to 0 for ALCE (Gao et al., 2023) due to citation requirements. We speed
up inference using vllm (Kwon et al., 2023). At each segment level, we adopt a beam width of 2.
For a token-level generation, we use greedy decoding. By default, we use the top five documents
from Contriever-MS MARCO (Izacard et al., 2022a); for biographies and open-domain QA, we
use additional top five documents retrieved by a web search engine, following Luo et al. (2023);
for ASQA, we use the author-provided top 5 documents by GTR-XXL (Ni et al., 2022) across all
baselines for a fair comparison.
5https://github.com/princeton-nlp/ALCE
6We report numbers using the results reported in the paper as the implementations are not available.
7
Preprint.
Table 2: Overall experiment results on six tasks. Bold numbers indicate the best performance among
non-proprietary models, and gray-colored bold text indicates the best proprietary model when
they outperforms all non-proprietary models.∗indicates concurrent or recent results reported by
concurrent work. – indicates numbers that are not reported by the original papers or are not applicable.
Models are sorted based on scale. FS, em, rg, mau, prec, rec denote FactScore (factuality); str-em,
rouge (correctness); MAUVE (fluency); citation precision and recall, respectively.
LM Short-form Closed-set Long-form generations (with citations)
PopQA TQA Pub ARC Bio ASQA
(acc) (acc) (acc) (acc) (FS) (em) (rg) (mau) (pre) (rec)
LMs with proprietary data
Llama2-c13B 20.0 59.3 49.4 38.4 55.9 22.4 29.6 28.6 – –
Ret-Llama2-c13B Ret-ChatGPT Perplexity.ai 51.8 59.8 52.1 37.9 79.9 32.8 34.8 43.8 19.8 36.1
ChatGPT 29.3 74.3 70.1 75.3 71.8 35.3 36.2 68.8 – –
50.8 65.7 54.7 75.3– 40.7 39.9 79.7 65.1 76.6
– – – – 71.2 – – – – –
Baselines without retrieval
Llama27B 14.7 30.5 34.2 21.8 44.5 7.9 15.3 19.0 – –
Alpaca7B 23.6 54.5 49.8 45.0 45.8 18.8 29.4 61.7 – –
Llama213B 14.7 38.5 29.4 29.4 53.4 7.2 12.4 16.0 – –
Alpaca13B 24.4 61.3 55.5 54.9 50.2 22.9 32.0 70.6 – –
CoVE65B * – – – – 71.2 – – – – –
Baselines with retrieval
Toolformer*6B – 48.8 – – – – – – – –
Llama27B Alpaca7B SAIL*7B Llama213B Alpaca13B 38.2 42.5 30.0 48.0 78.0 15.2 22.1 32.0 2.9 4.0
46.7 64.1 40.2 48.0 76.6 30.9 33.3 57.9 5.5 7.2
Llama2-FT7B 48.7 57.3 64.3 65.8 78.2 31.0 35.8 51.2 5.0 7.5
– – 69.2 48.4 – – – – – –
45.7 47.0 30.2 26.0 77.5 16.3 20.5 24.7 2.3 3.6
46.1 66.9 51.1 57.6 77.7 34.8 36.7 56.6 2.0 3.8
Our SELF-RAG 7B 54.9 66.4 72.4 67.3 81.2 30.0 35.7 74.3 66.9 67.8
Our SELF-RAG 13B 55.8 69.3 74.5 73.1 80.2 31.7 37.0 71.6 70.3 71.3
5 RESULTS AND ANALYSIS
5.1 MAIN RESULTS
Comparison against baselines without retrieval. Table 2 (top) presents the baselines without
retrieval. Our SELF-RAG (bottom two rows) demonstrates a substantial performance advantage
over supervised fine-tuned LLMs in all tasks and even outperforms ChatGPT in PubHealth, PopQA,
biography generations, and ASQA (Rouge and MAUVE). Our approach also significantly outperforms
a concurrent method that employs sophisticated prompt engineering; specifically, on the bio generation
task, our 7B and 13B models outperform the concurrent CoVE (Dhuliawala et al., 2023), which
iteratively prompts Llama265B to refine output.
Comparison against baselines with retrieval. As shown in Tables 2 (bottom), our SELF-RAG also
outperforms existing RAG in many tasks, obtaining the best performance among non-proprietary
LM-based models on all tasks. While our method outperforms other baselines, on PopQA or Bio,
powerful instruction-tuned LMs with retrieval (e.g., LLama2-chat, Alpaca) show large gains from
their non-retrieval baselines. However, we found that these baselines provide limited solutions for
tasks where we cannot simply copy or extract sub-strings of retrieved passages. On PubHealth
and ARC-Challenge, baselines with retrieval do not improve performance notably from their no-
retrieval counterparts. We also observe that most baselines with retrieval struggle to improve citation
accuracy. On ASQA, our model shows significantly higher citation precision and recall than all
models except ChatGPT. Gao et al. (2023) found that ChatGPT consistently exhibits superior efficacy
in this particular task, surpassing smaller LMs. Our SELF-RAG bridges this performance gap, even
outperforming ChatGPT in citation precision, which measures whether the model-generated claim is
fully supported by cited evidence. We also found that on the metrics for factual precision, SELF-RAG
7B occasionally outperforms our 13B due to the tendency of smaller SELF-RAG to often generate
8
Preprint.
PubHealth
PQA Med AS
(acc) (acc) (em)
70.5
SELF-RAG (50k) 45.5 73.5 32.1
Precision
70.0
Accuracy
Training
No Retriever R 43.6 67.8 31.0
No Critic C 42.6 72.0 18.1
Test
No retrieval 24.7 73.0 –
Hard constraints 28.3 72.6 –
Retrieve top1 41.8 73.1 28.6
Remove ISSUP 44.1 73.2 30.6
1 2
95
Mauve
90
Accuracy
1.00
0.99
0.99
0.98
1.0
0.8
0.6
0.0 0.2 0.4 0.6
PopQA
1.0
0.5
Frequency
0.0
1.00
0.75
0.50
Frequency
0.25
1 2
Weight for IsSupport
0.0 0.2 0.4 0.6
Retrieval Threshold
(a) Ablation
(b) Customization
(c) Retrieval
Figure 3: Analysis on SELF-RAG: (a) Ablation studies for key components of SELF-RAG training
and inference based on our 7B model. (b) Effects of soft weights on ASQA citation precision and
Mauve (fluency). (c) Retrieval frequency and normalized accuracy on PubHealth and PopQA.
precisely grounded yet shorter outputs. Llama2-FT7B, which is the baseline LM trained on the same
instruction-output pairs as SELF-RAG without retrieval or self-reflection and is retrieval-augmented
at test time only, lags behind SELF-RAG. This result indicates SELF-RAG gains are not solely from
training data and demonstrate the effectiveness of SELF-RAG framework.
5.2 ANALYSIS
Ablation studies. We conduct a set of ablations of our framework to identify which factors play
key roles. We evaluate two model variants trained differently than our model: No Retriever trains an
LM using the standard instruction-following method given instruction-output pairs, without retrieved
passages; No Critic trains an LM trained with input-output pairs that are always augmented with the
top one retrieved document without reflection tokens. This is similar to SAIL (Luo et al., 2023), and
we use our instruction-output data instead of using the Alpaca dataset (Dubois et al., 2023), as in
SAIL. We also conduct ablation on our inference-time algorithm, including No retrieval disables
retrieval during inference; Hard constraints indicates the model performance that retrieves when
Retrieve =Yes instead of using the adaptive threshold; Retrieve top 1 always retrieves and uses the
top one document only, similar to standard RAG approaches; Remove ISSUP indicates the model
performance that removes ISSUP score only during critique-guided beam search in Eq. 4. In this
ablation experiment, we use a training instance size of 50k for a more efficient exploration of training
variations. Later in this section, we conduct an analysis of the effect of training data size. We conduct
the ablation studies on three datasets, PopQA, PubHealth, and ASQA. On ASQA, we evaluate models
on sampled 150 instances and exclude ablations involving adaptive or no retrieval processes.
We show in Table 3a the ablation results. The top part of the table shows results for training ablations,
and the bottom part is for inference ablations. We see that all components play important roles. We
also observe a large performance gap between SELF-RAG and No Retriever or Critic baselines across
tasks, indicating that training an LM with those models largely contributes to the performance gain of
SELF-RAG. Using the top passages regardless of their relevance (Retrieve top 1) as in conventional
RAG approaches causes a large drop in PopQA and ASQA, and removing ISSUP during the beam
search results hurts performance on ASQA. This demonstrates the effectiveness of SELF-RAG’s
capabilities of carefully selecting generations based fine-grained multiple criterion, instead of naively
using all of the top passages from the retrieval model or solely depending on relevance scores.
Effects of inference-time customization. One key benefit of our proposed framework is that it
enables us to control how much each critique type affects the final generation sampling. We analyze
the effects of different parameter weights on the top of our 7B model during inference time on
ASQA, where multiple evaluation aspects are considered. Figure 3b shows the effects of changing
the weighting term for ISSUP , which criticizes how supported the output is by the text passage. As
the figure shows, increasing the weight leads to positive effects on the models’ citation precision
since this puts more emphasis on whether model generation is supported by the evidence. On the
9
Preprint.
Perfomance
55
50
45
40
35
73
72
71
Pop Bio.
60
S & P 92.5 70.0
ISREL 40
ISSUP 95.0 90.0
90.0 85.0
0 50 100 150
Num of training (k)
0 100
Num of training (k)
0 100
Num of training (k)
(a) PopQA
(b) PubHealth
(c) ASQA (prec)
(d) Human evaluation on PopQA
and Bio generation.
Figure 4: Training scale and Human analysis: (a) (b) (c) Training scale analysis shows the effect
of the training data scale on PopQA, PubHealth and ASQA (citation precision), respectively. (d)
Human analysis on SELF-RAG outputs as well as reflection tokens.
contrary, a larger weight results in lower MAUVE scores: when generation gets longer and more
fluent, there are often more claims that are not fully supported by citations, consistent with findings
by Liu et al. (2023a). Our framework lets practitioners choose and customize models’ behaviors at
test time by adjusting such parameters without requiring additional training.
Efficiency and accuracy trade-off. Using our framework, practitioners can adjust how often retrieval
occurs using the token probability of reward tokens. We evaluate how this adaptive threshold affects
overall accuracy and frequency of retrieval, and we evaluate the performance with varying numbers
of threshold δ (larger δ results in less retrieval) on PubHealth and PopQA. Figure 3c shows that
the model’s retrieval frequencies dramatically change on both datasets. as δvaries. On one hand,
performance deterioration by retrieving less is smaller on PubHealth but larger in PopQA.
Effects of training data size. We conduct an analysis of how the data scale affects the model’s
performance. In particular, we randomly sample 5k, 10k, 20k, and 50k instances from our original
150k training instances, and fine-tune four SELF-RAG 7B variants on those subsets. Then, we compare
the model performance on PopQA, PubHealth, and ASQA (citation precision) with our final SELF-
RAG trained on the full 150k instances. We also evaluate Figures 4a, 4b and 4c shows the models’
performance trained on different amount of data. Across all datasets, increasing data size often shows
upward trajectories and the improvements are significantly larger in PopQA and ASQA, while we do
not observed such significant improvements on Llama2-FT7B when increasing the training data from
50k to 150k. These results also indicate that further expanding the training data of SELF-RAG may
lead to further improvements, although in this work we limit our training data size to 150k.
Human evaluations. We conduct small human evaluations on SELF-RAG outputs, as well as the
reliability of predicted reflection tokens. In particular, we sampled 50 samples from PopQA and Bio
results. Following Menick et al. (2022), human annotators evaluate S&P, which indicates whether
the model output is plausible (i.e., the output is a reasonable and on-topic response to the question
as if it were occurring in a conversation) and supported (i.e., the provided evidence is sufficient to
verify the validity of the answer). For S&P, we do not consider the instances where SELF-RAG
predicts irrelevant or no support. We then ask our annotators whether the model-predicted
reflection tokens about ISREL and ISSUP match their inspections (e.g., whether the fully supported
output is supported by the cited evidence). Human annotators find SELF-RAG answers are often
plausible and supported by relevant passages with higher S&P scores on short-form PopQA, which is
consistent with Menick et al. (2022). Human annotators also find ISREL and ISSUP reflection token
predictions are mostly aligned with their assessments. Appendix Table 6 shows several annotated
examples and explanations on assessments.
6 CONCLUSION
This work introduces SELF-RAG, a new framework to enhance the quality and factuality of LLMs
through retrieval on demand and self-reflection. SELF-RAG trains an LM to learn to retrieve, generate,
and critique text passages and its own generation by predicting the next tokens from its original
vocabulary as well as newly added special tokens, called reflection tokens. SELF-RAG further enables
the tailoring of LM behaviors at test time by leveraging reflection tokens. Our holistic evaluations on
six tasks using multiple metrics demonstrate that SELF-RAG significantly outperforms LLMs with
more parameters or with conventional retrieval-augmented generation approaches.
10
Preprint.
ETHICAL CONCERNS
This work aims to improve the factuality of LLM outputs, the lack of which continues to cause nu-
merous real-world problems (e.g., spread of misinformation and provision of incorrect and dangerous
advice). While our method shows significant improvements in terms of performance, factuality, and
citation accuracy, it can still generate outputs that are not fully supported by the citations. We hope
that explicit self-reflection and fine-grained attribution may help users verify factual errors in the
model outputs.
ACKNOWLEDGMENTS
We thank Sewon Min, Scott Wen-tau Yih, Sean Welleck, and Kawin Ethayarajh for fruitful discussions
in the early stages of this work. We thank Sewon Min, Joongwon (Daniel) Kim, and Sandy Kaplan
for valuable feedback on the paper, and Tianyu Gao and Weijia Shi for their help on evaluations.
Akari Asai is supported by the IBM Fellowship. We thank Stability AI for providing computing
to train and evaluate the LMs in this work, and Microsoft Accelerate Foundation Models Research
Program for the access to OpenAI APIs. This work was funded in part by the DARPA MCS program
through NIWC Pacific (N66001-19-2-4031), NSF IIS-2044660, and gifts from AI2.
REFERENCES
Akari Asai, Kazuma Hashimoto, Hannaneh Hajishirzi, Richard Socher, and Caiming Xiong. Learn-
ing to retrieve reasoning paths over wikipedia graph for question answering. In International
Conference on Learning Representations, 2020. URL https://openreview.net/forum?
id=SJgVHkrYDH.
Akari Asai, Sewon Min, Zexuan Zhong, and Danqi Chen. Retrieval-based language models and appli-
cations. In Proceedings of the 61st Annual Meeting of the Association for Computational Linguistics
(Tutorial), 2023a. URL https://aclanthology.org/2023.acl-tutorials.6.
Akari Asai, Timo Schick, Patrick Lewis, Xilun Chen, Gautier Izacard, Sebastian Riedel, Hannaneh
Hajishirzi, and Wen-tau Yih. Task-aware retrieval with instructions. In Findings of the Associ-
ation for Computational Linguistics, 2023b. URL https://aclanthology.org/2023.
findings-acl.225.
Bernd Bohnet, Vinh Q Tran, Pat Verga, Roee Aharoni, Daniel Andor, Livio Baldini Soares, Jacob
Eisenstein, Kuzman Ganchev, Jonathan Herzig, Kai Hui, et al. Attributed question answering:
Evaluation and modeling for attributed large language models. arXiv preprint arXiv:2212.08037,
2022. URL https://arxiv.org/abs/2212.08037.
Lingjiao Chen, Matei Zaharia, and James Zou. How is chatgpt’s behavior changing over time? arXiv
preprint arXiv:2307.09009, 2023. URL https://arxiv.org/abs/2307.09009.
Peter Clark, Isaac Cowhey, Oren Etzioni, Tushar Khot, Ashish Sabharwal, Carissa Schoenick, and
Oyvind Tafjord. Think you have solved question answering? try arc, the ai2 reasoning challenge.
arXiv preprint arXiv:1803.05457, 2018. URL https://arxiv.org/abs/1803.05457.
Tri Dao, Dan Fu, Stefano Ermon, Atri Rudra, and Christopher R´ e. Flashattention: Fast and memory-
efficient exact attention with io-awareness. In Advances in Neural Information Processing Systems,
2022. URL https://openreview.net/forum?id=H4DqfPSibmx.
Shehzaad Dhuliawala, Mojtaba Komeili, Jing Xu, Roberta Raileanu, Xian Li, Asli Celikyilmaz, and
Jason Weston. Chain-of-verification reduces hallucination in large language models. arXiv preprint
arXiv:2309.11495, 2023. URL https://arxiv.org/abs/2309.11495.
Emily Dinan, Stephen Roller, Kurt Shuster, Angela Fan, Michael Auli, and Jason Weston. Wizard of
wikipedia: Knowledge-powered conversational agents. In International Conference on Learning
Representations, 2019. URL https://openreview.net/forum?id=r1l73iRqKm.
Yann Dubois, Xuechen Li, Rohan Taori, Tianyi Zhang, Ishaan Gulrajani, Jimmy Ba, Carlos Guestrin,
Percy Liang, and Tatsunori B. Hashimoto. Alpacafarm: A simulation framework for methods that
11
Preprint.
learn from human feedback. arXiv preprint arXiv:2305.14387, 2023. URL https://arxiv.
org/abs/2305.14387.
Tianyu Gao, Howard Yen, Jiatong Yu, and Danqi Chen. Enabling large language models to generate
text with citations. arXiv preprint arXiv:2305.14627, 2023. URL https://arxiv.org/abs/
2305.14627.
Kelvin Guu, Kenton Lee, Zora Tung, Panupong Pasupat, and Mingwei Chang. Retrieval augmented
language model pre-training. In International Conference on Machine Learning, 2020. URL
https://dl.acm.org/doi/pdf/10.5555/3524938.3525306.
Gautier Izacard, Mathilde Caron, Lucas Hosseini, Sebastian Riedel, Piotr Bojanowski, Armand
Joulin, and Edouard Grave. Unsupervised dense information retrieval with contrastive learning.
Transactions on Machine Learning Research, 2022a. URL https://openreview.net/
forum?id=jKN1pXi7b0.
Gautier Izacard, Patrick Lewis, Maria Lomeli, Lucas Hosseini, Fabio Petroni, Timo Schick, Jane
Dwivedi-Yu, Armand Joulin, Sebastian Riedel, and Edouard Grave. Few-shot learning with
retrieval augmented language models. arXiv preprint arXiv:2208.03299, 2022b. URL https:
//arxiv.org/abs/2208.03299.
Zhengbao Jiang, Frank F Xu, Luyu Gao, Zhiqing Sun, Qian Liu, Jane Dwivedi-Yu, Yiming Yang,
Jamie Callan, and Graham Neubig. Active retrieval augmented generation. arXiv preprint
arXiv:2305.06983, 2023. URL https://arxiv.org/abs/2305.06983.
Mandar Joshi, Eunsol Choi, Daniel Weld, and Luke Zettlemoyer. TriviaQA: A large scale distantly
supervised challenge dataset for reading comprehension. In Proceedings of the 55th Annual
Meeting of the Association for Computational Linguistics (Volume 1: Long Papers), 2017. URL
https://aclanthology.org/P17-1147.
Nitish Shirish Keskar, Bryan McCann, Lav R Varshney, Caiming Xiong, and Richard Socher.
Ctrl: A conditional transformer language model for controllable generation. arXiv preprint
arXiv:1909.05858, 2019. URL https://arxiv.org/abs/1909.05858.
Tomasz Korbak, Kejian Shi, Angelica Chen, Rasika Vinayak Bhalerao, Christopher Buckley, Jason
Phang, Samuel R Bowman, and Ethan Perez. Pretraining language models with human preferences.
In International Conference on Machine Learning, 2023. URL https://openreview.net/
forum?id=AT8Iw8KOeC.
Tom Kwiatkowski, Jennimaria Palomaki, Olivia Redfield, Michael Collins, Ankur Parikh, Chris
Alberti, Danielle Epstein, Illia Polosukhin, Jacob Devlin, Kenton Lee, Kristina Toutanova, Llion
Jones, Matthew Kelcey, Ming-Wei Chang, Andrew M. Dai, Jakob Uszkoreit, Quoc Le, and
Slav Petrov. Natural questions: A benchmark for question answering research. Transactions of
the Association for Computational Linguistics, 2019. URL https://aclanthology.org/
Q19-1026.
Woosuk Kwon, Zhuohan Li, Siyuan Zhuang, Ying Sheng, Lianmin Zheng, Cody Hao Yu, Joseph E.
Gonzalez, Hao Zhang, and Ion Stoica. Efficient memory management for large language model
serving with pagedattention. In Proceedings of the ACM SIGOPS 29th Symposium on Operating
Systems Principles, 2023. URL https://arxiv.org/abs/2309.06180.
Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin, Naman Goyal,
Heinrich K¨ uttler, Mike Lewis, Wen-tau Yih, Tim Rockt¨ aschel, Sebastian Riedel, and Douwe Kiela.
Retrieval-augmented generation for knowledge-intensive nlp tasks. In Advances in Neural Infor-
mation Processing Systems, 2020. URL https://proceedings.neurips.cc/paper/
2020/file/6b493230205f780e1bc26945df7481e5-Paper.pdf.
Xi Victoria Lin, Xilun Chen, Mingda Chen, Weijia Shi, Maria Lomeli, Rich James, Pedro Rodriguez,
Jacob Kahn, Gergely Szilvasy, Mike Lewis, Luke Zettlemoyer, and Scott Yih. Ra-dit: Retrieval-
augmented dual instruction tuning, 2023. URL https://arxiv.org/abs/2310.01352.
Nelson F Liu, Tianyi Zhang, and Percy Liang. Evaluating verifiability in generative search engines.
arXiv preprint arXiv:2304.09848, 2023a. URL https://arxiv.org/abs/2304.09848.
12
Preprint.
Yang Liu, Dan Iter, Yichong Xu, Shuohang Wang, Ruochen Xu, and Chenguang Zhu. Gpteval: Nlg
evaluation using gpt-4 with better human alignment. arXiv preprint arXiv:2303.16634, 2023b.
URL https://arxiv.org/abs/2303.16634.
Ximing Lu, Sean Welleck, Jack Hessel, Liwei Jiang, Lianhui Qin, Peter West, Prithviraj Am-
manabrolu, and Yejin Choi. QUARK: Controllable text generation with reinforced unlearning.
In Advances in Neural Information Processing Systems, 2022. URL https://openreview.
net/forum?id=5HaIds3ux5O.
Hongyin Luo, Yung-Sung Chuang, Yuan Gong, Tianhua Zhang, Yoon Kim, Xixin Wu, Danny Fox,
Helen Meng, and James Glass. Sail: Search-augmented instruction learning. arXiv preprint
arXiv:2305.15225, 2023. URL https://arxiv.org/abs/2305.15225.
Aman Madaan, Niket Tandon, Prakhar Gupta, Skyler Hallinan, Luyu Gao, Sarah Wiegreffe, Uri
Alon, Nouha Dziri, Shrimai Prabhumoye, Yiming Yang, Shashank Gupta, Bodhisattwa Prasad
Majumder, Katherine Hermann, Sean Welleck, Amir Yazdanbakhsh, and Peter Clark. Self-
refine: Iterative refinement with self-feedback. arXiv preprint arXiv:2303.17651, 2023. URL
https://arxiv.org/abs/2303.17651.
Alex Mallen, Akari Asai, Victor Zhong, Rajarshi Das, Daniel Khashabi, and Hannaneh Hajishirzi.
When not to trust language models: Investigating effectiveness of parametric and non-parametric
memories. In Proceedings of the 61st Annual Meeting of the Association for Computational
Linguistics (Volume 1: Long Papers), 2023. URL https://aclanthology.org/2023.
acl-long.546.
Jacob Menick, Maja Trebacz, Vladimir Mikulik, John Aslanides, Francis Song, Martin Chadwick,
Mia Glaese, Susannah Young, Lucy Campbell-Gillingham, Geoffrey Irving, et al. Teaching
language models to support answers with verified quotes. arXiv preprint arXiv:2203.11147, 2022.
URL https://arxiv.org/abs/2203.11147.
Todor Mihaylov, Peter Clark, Tushar Khot, and Ashish Sabharwal. Can a suit of armor conduct
electricity? a new dataset for open book question answering. In Proceedings of the 2018 Conference
on Empirical Methods in Natural Language Processing, 2018. URL https://aclanthology.
org/D18-1260.
Sewon Min, Danqi Chen, Hannaneh Hajishirzi, and Luke Zettlemoyer. A discrete hard EM approach
for weakly supervised question answering. In Proceedings of the 2019 Conference on Empirical
Methods in Natural Language Processing and the 9th International Joint Conference on Natu-
ral Language Processing (EMNLP-IJCNLP), 2019. URL https://aclanthology.org/
D19-1284.
Sewon Min, Kalpesh Krishna, Xinxi Lyu, Mike Lewis, Wen-tau Yih, Pang Wei Koh, Mohit Iyyer,
Luke Zettlemoyer, and Hannaneh Hajishirzi. Factscore: Fine-grained atomic evaluation of factual
precision in long form text generation. arXiv preprint arXiv:2305.14251, 2023. URL https:
//arxiv.org/abs/2305.14251.
Reiichiro Nakano, Jacob Hilton, Suchir Balaji, Jeff Wu, Long Ouyang, Christina Kim, Christopher
Hesse, Shantanu Jain, Vineet Kosaraju, William Saunders, et al. Webgpt: Browser-assisted
question-answering with human feedback. arXiv preprint arXiv:2112.09332, 2021. URL https:
//arxiv.org/abs/2112.09332.
Jianmo Ni, Chen Qu, Jing Lu, Zhuyun Dai, Gustavo Hernandez Abrego, Ji Ma, Vincent Zhao,
Yi Luan, Keith Hall, Ming-Wei Chang, and Yinfei Yang. Large dual encoders are generalizable
retrievers. In Proceedings of the 2022 Conference on Empirical Methods in Natural Language
Processing, 2022. URL https://aclanthology.org/2022.emnlp-main.669.
OpenAI. Gpt-4 technical report. arXiv preprint arXiv:2303.08774, 2023. URL https://arxiv.
org/abs/2303.08774.
Long Ouyang, Jeffrey Wu, Xu Jiang, Diogo Almeida, Carroll Wainwright, Pamela Mishkin, Chong
Zhang, Sandhini Agarwal, Katarina Slama, Alex Gray, John Schulman, Jacob Hilton, Fraser Kelton,
Luke Miller, Maddie Simens, Amanda Askell, Peter Welinder, Paul Christiano, Jan Leike, and
13
Preprint.
Ryan Lowe. Training language models to follow instructions with human feedback. In Advances in
Neural Information Processing Systems, 2022. URL https://openreview.net/forum?
id=TG8KACxEON.
Debjit Paul, Mete Ismayilzada, Maxime Peyrard, Beatriz Borges, Antoine Bosselut, Robert West,
and Boi Faltings. Refiner: Reasoning feedback on intermediate representations. arXiv preprint
arXiv:2304.01904, 2023. URL https://arxiv.org/abs/2304.01904.
Fabio Petroni, Aleksandra Piktus, Angela Fan, Patrick Lewis, Majid Yazdani, Nicola De Cao, James
Thorne, Yacine Jernite, Vladimir Karpukhin, Jean Maillard, Vassilis Plachouras, Tim Rockt¨ aschel,
and Sebastian Riedel. KILT: a benchmark for knowledge intensive language tasks. In Proceedings
of the 2021 Conference of the North American Chapter of the Association for Computational
Linguistics: Human Language Technologies, 2021. URL https://aclanthology.org/
2021.naacl-main.200.
Krishna Pillutla, Swabha Swayamdipta, Rowan Zellers, John Thickstun, Sean Welleck, Yejin Choi,
and Zaid Harchaoui. MAUVE: Measuring the gap between neural text and human text using
divergence frontiers. In Advances in Neural Information Processing Systems, 2021. URL https:
//openreview.net/forum?id=Tqx7nJp7PR.
Samyam Rajbhandari, Jeff Rasley, Olatunji Ruwase, and Yuxiong He. Zero: Memory optimizations
toward training trillion parameter models. In Proceedings of the International Conference for High
Performance Computing, Networking, Storage and Analysis, 2020. URL https://dl.acm.
org/doi/10.5555/3433701.3433727.
Ori Ram, Yoav Levine, Itay Dalmedigos, Dor Muhlgay, Amnon Shashua, Kevin Leyton-Brown, and
Yoav Shoham. In-context retrieval-augmented language models. Transactions of the Association
for Computational Linguistics, 2023. URL https://arxiv.org/abs/2302.00083.
Victor Sanh, Albert Webson, Colin Raffel, Stephen Bach, Lintang Sutawika, Zaid Alyafeai, Antoine
Chaffin, Arnaud Stiegler, Arun Raja, Manan Dey, M Saiful Bari, Canwen Xu, Urmish Thakker,
Shanya Sharma Sharma, Eliza Szczechla, Taewoon Kim, Gunjan Chhablani, Nihal Nayak, De-
bajyoti Datta, Jonathan Chang, Mike Tian-Jian Jiang, Han Wang, Matteo Manica, Sheng Shen,
Zheng Xin Yong, Harshit Pandey, Rachel Bawden, Thomas Wang, Trishala Neeraj, Jos Rozen,
Abheesht Sharma, Andrea Santilli, Thibault Fevry, Jason Alan Fries, Ryan Teehan, Teven Le Scao,
Stella Biderman, Leo Gao, Thomas Wolf, and Alexander M Rush. Multitask prompted training
enables zero-shot task generalization. In International Conference on Learning Representations,
2022. URL https://openreview.net/forum?id=9Vrb9D0WI4.
Timo Schick, Jane Dwivedi-Yu, Roberto Dess` ı, Roberta Raileanu, Maria Lomeli, Luke Zettlemoyer,
Nicola Cancedda, and Thomas Scialom. Toolformer: Language models can teach themselves to
use tools. arXiv preprint arXiv:2302.04761, 2023. URL https://arxiv.org/abs/2302.
04761.
John Schulman, Filip Wolski, Prafulla Dhariwal, Alec Radford, and Oleg Klimov. Proximal policy
optimization algorithms. arXiv preprint arXiv:1707.06347, 2017. URL https://arxiv.org/
abs/1707.06347.
Freda Shi, Xinyun Chen, Kanishka Misra, Nathan Scales, David Dohan, Ed H. Chi, Nathanael
Sch¨ arli, and Denny Zhou. Large language models can be easily distracted by irrelevant context.
In Proceedings of the 40th International Conference on Machine Learning, 2023. URL https:
//proceedings.mlr.press/v202/shi23a.html.
Ivan Stelmakh, Yi Luan, Bhuwan Dhingra, and Ming-Wei Chang. ASQA: Factoid questions meet long-
form answers. In Proceedings of the 2022 Conference on Empirical Methods in Natural Language
Processing, 2022. URL https://aclanthology.org/2022.emnlp-main.566.
James Thorne, Andreas Vlachos, Christos Christodoulopoulos, and Arpit Mittal. FEVER: a large-
scale dataset for fact extraction and VERification. In Proceedings of the 2018 Conference of the
North American Chapter of the Association for Computational Linguistics: Human Language Tech-
nologies, Volume 1 (Long Papers), 2018. URL https://aclanthology.org/N18-1074.
14
Preprint.
Hugo Touvron, Louis Martin, Kevin Stone, Peter Albert, Amjad Almahairi, Yasmine Babaei, Nikolay
Bashlykov, Soumya Batra, Prajjwal Bhargava, Shruti Bhosale, et al. Llama 2: Open foundation
and fine-tuned chat models. arXiv preprint arXiv:2307.09288, 2023. URL https://arxiv.
org/abs/2307.09288.
Yizhong Wang, Hamish Ivison, Pradeep Dasigi, Jack Hessel, Tushar Khot, Khyathi Raghavi Chandu,
David Wadden, Kelsey MacMillan, Noah A Smith, Iz Beltagy, et al. How far can camels go?
exploring the state of instruction tuning on open resources. arXiv preprint arXiv:2306.04751, 2023.
URL https://arxiv.org/abs/2306.04751.
Jason Wei, Maarten Bosma, Vincent Zhao, Kelvin Guu, Adams Wei Yu, Brian Lester, Nan Du,
Andrew M. Dai, and Quoc V Le. Finetuned language models are zero-shot learners. In International
Conference on Learning Representations, 2022. URL https://openreview.net/forum?
id=gEZrGCozdqR.
Zeqiu Wu, Yushi Hu, Weijia Shi, Nouha Dziri, Alane Suhr, Prithviraj Ammanabrolu, Noah A
Smith, Mari Ostendorf, and Hannaneh Hajishirzi. Fine-grained human feedback gives better
rewards for language model training. arXiv preprint arXiv:2306.01693, 2023. URL https:
//arxiv.org/abs/2306.01693.
Yuxi Xie, Kenji Kawaguchi, Yiran Zhao, Xu Zhao, Min-Yen Kan, Junxian He, and Qizhe Xie. Decom-
position enhances reasoning via self-evaluation guided decoding. arXiv preprint arXiv:2305.00633,
2023. URL https://arxiv.org/abs/2305.00633.
Fangyuan Xu, Weijia Shi, and Eunsol Choi. Recomp: Improving retrieval-augmented lms with
compression and selective augmentation, 2023. URL https://arxiv.org/abs/2310.
04408.
Ori Yoran, Tomer Wolfson, Ori Ram, and Jonathan Berant. Making retrieval-augmented language
models robust to irrelevant context, 2023. URL https://arxiv.org/abs/2310.01558.
Xiang Yue, Boshi Wang, Kai Zhang, Ziru Chen, Yu Su, and Huan Sun. Automatic evaluation of
attribution by large language models. arXiv preprint arXiv:2305.06311, 2023. URL https:
//arxiv.org/abs/2305.06311.
Tianhua Zhang, Hongyin Luo, Yung-Sung Chuang, Wei Fang, Luc Gaitskell, Thomas Hartvigsen,
Xixin Wu, Danny Fox, Helen Meng, and James Glass. Interpretable unified language checking.
arXiv preprint arXiv:2304.03728, 2023. URL https://arxiv.org/abs/2304.03728.
Andy Zhou, Kai Yan, Michal Shlapentokh-Rothman, Haohan Wang, and Yu-Xiong Wang. Language
agent tree search unifies reasoning acting and planning in language models, 2023. URL https:
//arxiv.org/abs/2310.04406.
Daniel M Ziegler, Nisan Stiennon, Jeffrey Wu, Tom B Brown, Alec Radford, Dario Amodei, Paul
Christiano, and Geoffrey Irving. Fine-tuning language models from human preferences. arXiv
preprint arXiv:1909.08593, 2019. URL https://arxiv.org/abs/1909.08593.
15
Preprint.
APPENDIX
A SELF-RAG Details 17
A.1 Reflection Tokens. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . A.2 SELF-RAG Training . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . A.3 SELF-RAG Inference . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 17
17
19
B Experimental Details 19
B.1 More Details of Training . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . B.2 More Details of Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19
20
C Results 20
C.1 Analysis . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . C.2 Human Evaluation Examples . . . . . . . . . . . . . . . . . . . . . . . . . . . . . C.3 Qualitative Examples . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . D Full List of Instructions and Demonstrations for GPT-4 20
21
21
21
16
Preprint.
A SELF-RAG DETAILS
A.1 REFLECTION TOKENS.
Definitions of reflection tokens. Below, we provide a detailed definition of reflection type and
output tokens. The first three aspects will be provided at each segment level, while the final aspect is
only given at each output level.
• Retrieval-on-demand ( Retrieve ): Given an input and previous-step generation (if applicable),
an LM determines whether the continuation requires factual grounding. No indicates retrieval
is unnecessary as the sequence does not require factual grounding or may not be enhanced by
knowledge retrieval, Yes indicates retrieval is necessary. We additionally have continue
to use evidence, which indicates that a model can continue to use the evidence retrieved
previously. For instance, a passage may contain rich factual information, and thus SELF-RAG
generates multiple segments based on the passage.
• Relevant ( ISREL ): Retrieved knowledge may not be always relevant to the input. This aspect
indicates whether the evidence provides useful information (Relevant) or not (Irrelevant).
• Supported ( ISSUP ): Attribution is the concept of whether the output is fully supported by
certain evidence (Menick et al., 2022; Bohnet et al., 2022). This aspect judges how much infor-
mation in the output is entailed by the evidence. We evaluate attributions in three scale, Fully
supported, Partially supported, and No support / Contradictory, follow-
ing Yue et al. (2023); Nakano et al. (2021).
• Useful ( ISUSE ): Following the definitions from Liu et al. (2023a), we define the perceived utility
as whether the response is a helpful and informative answer to the query, independently from
whether it is in fact factual or not. This can be also viewed as plausibility in Menick et al. (2022).
For usefulness, we use a five-scale evaluation (1 is the lowest and 5 is the highest).
Details of GPT-4-based data collections. We use the instruction and demonstration pairs to prompt
GPT-4, listed in Section D. Following an official recommendation, we separate instructions and
outputs with “##”. We use the temperature 1 and set the maximum output token counts to be 200. We
discard instances where GPT-4 does not follow the designated output formats or output sequences
that do not match our expected category names. As a result, we collected 1,2594 for Retrieve , 11,181
for ISSUP , 19,317 for relevance, 3,831 for utility.
Manual analysis of the GPT-4 predictions. The authors of this paper manually assess randomly
sampled 20 instances for each aspect and check if GPT-4 predictions match their assessments given
the same instruction, demonstrations, and test instances. We found our assessments show high
agreement with GPT-4 predictions, especially for relevance (95%), retrieval necessity (95%), and
the degree of support (90%). Agreement was slightly lower in usefulness (80%), mostly due to the
disagreement between 1 and 2 or 4 and 5.
A.2 SELF-RAG TRAINING
Overview of training. Algorithm 2 provides a high-level overview of our training.
Full list of seed datasets. To sample diverse input-output pairs, we sample instances of the Open-
Instruct (Wang et al., 2023) dataset. In particular, we use their ShareGPT, GPT-4 Alpaca, Alpaca,
OpenAssistant, and FLAN subsets subsets. We also sample instances from a couple of knowledge-
intensive datasets, Natural Questions (Kwiatkowski et al., 2019), Wizard of Wikipedia (Dinan et al.,
2019) and FEVER (Thorne et al., 2018) from the KILT benchmark (Petroni et al., 2021), ASQA (Stel-
makh et al., 2022) and multiple QA datasets including ARC-Easy and OpenBookQA (Mihaylov et al.,
2018). Table 3 shows the full list of training instances, and in total, we use 145,619 instances.
Performance of the Critic C. We evaluate the accuracy of reward predictions by splitting GPT-4
generated feedback into training, development, and test sets. The accuracy of the reward model is
as follows. Table 5 shows the model performance of predicting GPT-4 judgments. As you can see,
overall our fine-tuned reward model shows high prediction matching with GPT-4 predicted feedback.
17
Preprint.
Algorithm 2 SELF-RAG Training
1: Input input-output data D= {X,Y}, generator M, Cθ
2: Initialize Cwith a pre-trained LM
3: Sample data {Xsample,Ysample}∼{X,Y} 4: for (x,y) ∈(Xsample,Ysample) do 5: Prompt GPT-4 to collect a reflection token rfor (x,y)
6: Add {(x,y,r)}to Dcritic
7: Update Cwith next token prediction loss 8: Initialize Mwith a pre-trained LM 9: for (x,y) ∈(X,Y) do 10: Run Cto predict rgiven (x,y)
11: Add (x,y,r) to Dgen
12: Update Mon Dgen with next token prediction loss ▷Training Critic LM (Section 3.2.1)
▷Data collections for C
▷Critic learning; Eq. 1
▷Training Generator LM (Section 3.2.2)
▷Data collection for Mwith Dcritic
▷Generator LM learning; Eq. 2
Dataset name category Data source the number of instances
GPT-4 Alpaca Stanford Alpaca FLAN-V2 ShareGPT Open Assistant 1 Wizard of Wikipedia Natural Questions FEVER OpenBoookQA Arc-Easy ASQA Instruction-following Open-Instruct 26,168
Instruction-following Open-Instruct 25,153
Instruction-following Open-Instruct 17,817
Instruction-following Open-Instruct 13,406
Instruction-following Open-Instruct 9,464
Knowledge-intensive KILT 17,367
Knowledge-intensive KILT 15,535
Knowledge-intensive KILT 9,966
Knowledge-intensive HF Dataset 4,699
Knowledge-intensive HF Dataset 2,147
Knowledge-intensive ASQA 3,897
Table 3: The generator LM Mtraining data statistics.
base LM Retrieve ISSUP ISREL ISUSE
Llama2-7B FLAN-3B 93.8 93.5 80.2 73.5
85.6 73.1 82.0 72.1
Figure 5: Reward prediction accuracy using GPT-4 predictions as ground-truth predictions.
While our final model uses Llama2-7B as a base LM, we also train and compare FLAN-3B (Wei
et al., 2022) model on the same data, to investigate the effectiveness of different data sizes affect final
reward predictions. In most aspects, our reward model shows higher than 80% accuracy, indicating
the powerful ability of fine-tuned specialized LMs to evaluate text. While both models show relatively
lower performance on ISUSE , this is because both models often confuse between the two highest
cases (5 and 4), where human annotators can also disagree.
Details of Mdata creation. Here, we provide detailed data creation procedures. Algorithm 3
summarizes the process. Here we set yt to yfor simplification. Once we train the critic model, we
first run it on input data from the aforementioned datasets, to predict whether retrieval is needed or
not. For the instances where the critic predicts Retrieve =No, we only predict the ISUSE given input
and output. For the instances where the critic predicts Retrieve =Yes, we first retrieve passages using
the input and the entire output as queries, to find passages that are relevant to the entire output. We
then split output sentences using Spacy.7 For each sentence, we run Cto predict whether the retrieval
is necessary or not, given the input, preceding segments, and the initial retrieved passage. If Cpredicts
Retrieve =No, then do not insert any paragraph at the tth segment. If Cpredicts Retrieve =Yes, then
we use the original input and the tth segment as a retrieval query to find relevant passages for the
t-th segment. For each retrieved passage, we predict ISREL and ISSUP . If there is any passage and
continuation with ISREL =Relevant and ISSUP =Fully Supported / ISSUP =Partially
7https://spacy.io/
18
Preprint.
Supported, then we sample it as the continuation. If there is more than one passage satisfying this
criterion, we use the one with the highest retrieval score. If there are only =Irrelevant or
ISSUP =No Support passages, we randomly sample one passage.
ISREL Algorithm 3 Mgen Data creation
1: Input Input-output data D= X,Y
2: for (x,y) ∈{X,Y}do
3: Given (x,y) Cpredicts Retrieve
4: if Retrieve is predicted then
5: Retrieve relevant passages D using Rgiven (x,y) 6: for d∈D do
7: Cpredicts ISREL for each d 8: Cpredicts ISSUP for each (y,d) 9: Cpredicts ISUSE for each d 10: Sample d
11: else if Retrieve is not predicted then
12: Cpredicts ISUSE given x,y
Add augmented (x,y,d,r) to Dgen
▷Retrieve passages
▷Predict relevance of passages
▷Predict supports of outputs
▷Predict overall utility (t= T only)
where S= t∈{1,2,3,4,5}p( ISUSE = t).
Details of adaptive retrieval. following condition is satisfied:
Training examples. Table 4 show several training examples used for Mtraining.
A.3 SELF-RAG INFERENCE
Details of beam-search score calculations. taking the normalized probabilities of desirable tokens. For We first compute scores for each critique type by
, we compute the score as follows:
ISREL s( ISREL ) = p( ISREL = RELEVANT)
p( ISREL = RELEVANT) + p( ISREL = IRRELEVANT).
For ISSUP , we compute the score as follows:
s( ISREL ) = p( ISSUP = FULLY)
S + 0.5 ×p( ISSUP = PARTIALLY)
S ,
where S= t∈{FULLY,PARTIALLY,NO}p( ISSUP = t). For ISUSE where we have a five-scale score, we
compute the weighted sum of the scores. We assigns weighted scores of w= {−1,−0.5,0,0.5,1}
to the tokens ISUSE ={1,2,3,4,5}, and compute the final scores as follows:
s( ISUSE ) =
5
p( ISUSE = i)
wi
S ,
i
For retrieval based on soft constraints, we trigger retrieval if the
p( Retrieve = YES)
p( Retrieve = YES) + p(p( Retrieve = NO) >δ.
B EXPERIMENTAL DETAILS
B.1 MORE DETAILS OF TRAINING
More details of training and computations. We use 4 Nvidia A100 with 80GB memory to train
our models. All models are trained for 3 epochs with a batch size of 128, a peak learning rate of 2e-5
with 3% warmup steps, and linear decay afterward. We set the maximum token length to be 2,048
for the 7B model, and 1,524 for the 13B model due to the memory constraint. We use Deepspeed
stage 3 (Rajbhandari et al., 2020) to conduct multi-GPU distributed training, with training precision
Bfloat16 enabled. FlashAttention (Dao et al., 2022) is used to make the long-context training more
efficient. We run inference of our trained models using 1-2 Quadro RTX 6000 GPUs with 24GB
memory.
19
Preprint.
B.2 MORE DETAILS OF EVALUATIONS
Retrieval setup details. By default, we use Contriever-MS MARCO to retrieve the top five
documents from Wikipedia, and use official Wikipedia embeddings based on 2018 English Wikipedia.
On PopQA, where question and answer pairs are created based on WikiData in 2022, we found
that the 2018 Wikipedia sometimes lacks articles about some entities that have been more recently
added to Wikipedia. Therefore, for PopQA, we used the December 2020 preprocessed Wikipedia
corpus provided by Izacard et al. (2022b) and generated document embeddings.8 The issues of
performance variance from different Wikipedia dumps have been reported by prior work (Asai et al.,
2020; Izacard et al., 2022b). Yet, we observe limited effectiveness of such off-the-shelf retrieval
models trained primarily on knowledge-intensive tasks for open-ended generation (e.g., instruction
following). Recent or concurrent work studies instruction-tuning of retrieval systems (Asai et al.,
2023b) or joint training of retrieval and LM components (Lin et al., 2023), while we leave exploring
the effectivess of such appraoches for future work. For bio generation and open-domain QA tasks,
we additionally retrieve five documents using Google Programmable Search9 and search documents
from English Wikipedia. As this API only provides snippets, we retrieve Wikipedia introductory
paragraphs for the corresponding entities.
Detailed experimental settings for individual datasets. For OpenQA datasets, we set the max-
imum new token number to 100 tokens. For closed-set tasks (PubHealth and ARC-C), we set the
maximum new token length to 50 for all baselines. For SELF-RAG inference on PubHealth and
ARC-C, instead of determining the output with the highest score 4 as in other tasks, we aggregate the
scores for each option and select the answer option with the highest score. We found in zero-shot
settings of fact checking, some LLMs can generate capitalized class labels (e.g., True) while our
gold labels are lower-cased. Therefore, across different LMs, for fact checking, we lowercase the
predictions. In multiple choice tasks, we found some models generate answers in slightly different
ways (e.g., (A) instead of A). We slightly modify instructions for each LLM to avoid such format
violations, and further conduct string matching between each candidate and model predictions if
format violations still remain. After that processing, in closed set tasks, model predictions match
one of the gold classes in almost all cases. For ALCE, we found that Llama2-chat tend to generate
significantly lower outputs than other models (e.g., on average, their output is nearly 100 token, while
ChatGPT generates 40 tokens on average), resulting in inflated str-em scores. We limit the maximum
generation length to 100 tokens for all baselines to avoid this issue, rather than the original 300
tokens in the ALCE paper. Consequently, all of the baseline output length is within 30-60 tokens.
For FactScore, we set the maximum new token length to 500 for baselines and 200 for SELF-RAG at
each segment level.
Task-specific instructions. Table 5 shows the list of the instructions used during evaluations. For
Open-domain QA, we do not provide explicit instructions.
C RESULTS
C.1 ANALYSIS
Reliance on parametric- and non-parametric memories. We conduct analysis on how frequently
model answers come from retrieved passages (non-parametric memories) or their own parametric
memories. On two open-domain QA datasets, TriviaQA and PopQA, we conduct the following
analysis: 1) sample query models successfully answer correctly, 2) for each query in this group,
check whether the matched ground-truth answer is a sub-string of the retrieved passage or not. We
evaluate SELF-RAG 7B, Alpaca 7B, Alpaca 13B, and Llama2-Chat-13B. We found that SELF-RAG
significantly less frequently generates answers that are not included in the provided evidence; in
particular, in Alpaca 30B, 20% of the correct predictions are not included in the provided passages,
followed by Llama2-chat 13B (18%) and Alpaca (15%), while it is only 2% in SELF-RAG. When
retrieved passages are not relevant, SELF-RAG generates ISREL =Irrelevant, indicating that the
following answers may not be factually grounded, while those instruction-tuned models continue to
generate plausible answers.
8https://github.com/facebookresearch/atlas
9https://programmablesearchengine.google.com/about/
20
Preprint.
ISREL and ISSUP
C.2 HUMAN EVALUATION EXAMPLES
Table 6 shows examples with human evaluations on S&P and correctness of reflection tokens.
C.3 QUALITATIVE EXAMPLES
Table 7 shows several examples predicted by our SELF-RAG (13B). The first example is the model
output to an ASQA question. The first reference states that Emperor Constantine made Sunday a
day of rest from labor, and further the second citation supports the fact that the official adoption
of Sunday as a day of rest by Constantine in AD 321. In the second example, the model predicts
Contradictory to the first output as the output says the person has served as the CEO since 2010,
while the passage says he stepped down as CEO in 2015. Indicating those factual contradictions
as reflection tokens enables to enforcement of hard control and also verification of model outputs
easily. In the third example, while the generation is mostly correct, SELF-RAG predicts Partially
Support to the statement listing the name of the songs, as they were not explicitly mentioned.
D FULL LIST OF INSTRUCTIONS AND DEMONSTRATIONS FOR GPT-4
Here, we show the instructions and demonstrations used to prompt GPT-4 to collect reflection tokens.
Table 8 shows the instructions and demonstrations for the initial retrieval token. Table 9 shows
the instruction and demonstrations used to collect the three-way output tokens for Retrieve given
instruction, preceding sentences, and previously retrieved passages. Due to the longer demonstration
and test input, we only use a single demonstration. Table 10 shows an instruction and demonstrations
used to collect the three-way output tokens for ISREL . Table 11 shows an instruction and demon-
strations used to collect the three-way output tokens for ISREL . Table 12 shows an instruction and
demonstrations used to collect the five-way output tokens for ISUSE .
21
Preprint.
Input Identify the two types of computer memory. (from gpt4 alpaca)
Output [ Retrieve =Yes]<p>Computer memory Computer memory – In computing, memory refers to the
computer hardware integrated circuits that store information for immediate use in a computer; it is synonymous
with the term “primary storage”. Computer memory operates at a high speed, for example random-access memory
(RAM), as a distinction from storage that provides slow-to-access information but offers higher capacities. If
needed, contents of the computer memory can be transferred to secondary storage; a very common way of doing
this is through a memory management technique called “virtual memory”. An archaic synonym for memory is
store. The term “memory”, meaning “primary storage” or “main memory”, is<p>[ ISREL =Relevant]Primary
memory, also known as main memory, is the memory that the computer can directly access and manipulate.
[ ISSUP =Fully Supported] [ Retrieve =Yes]<p>Computer memory–often associated with addressable
semiconductor memory, i.e. integrated circuits consisting of silicon-based transistors, used for example as
primary storage but also other purposes in computers and other digital electronic devices. There are two main
kinds of semiconductor memory, volatile and non-volatile. Examples of non-volatile memory are flash memory
(used as secondary memory) and ROM, PROM, EPROM and EEPROM memory (used for storing firmware
such as BIOS). Examples of volatile memory are primary storage, which is typically dynamic random-access
memory (DRAM), and fast CPU cache memory, which is typically static random-access memory (SRAM)
that is fast but energy-consuming, offering lower memory<p>[ ISREL =Relevant]Primary memory can be
accessed quickly, but it is volatile, meaning that the data is lost if the power is turned off.[ ISSUP =Partially
Supported]...[ ISUSE :5]
Input Give an example of a situation where you have to delegate tasks (from gpt4 alpaca)
Output [ Retrieve =No]The manager realizes that there is an extensive list of tasks that need to be completed in
order to make the event a success.[ Retrieve =Yes]<p>Delegation–Delegation is the assignment of any responsi-
bility or authority to another person (normally from a manager to a subordinate) to carry out specific activities. It
is one of the core concepts of management leadership. However, the person who delegated the work remains
accountable for the outcome of the delegated work. Delegation empowers a subordinate to make decisions, i.e. it
is a shifting of decision-making authority from one organizational level to a lower one. Delegation, if properly
done, is not fabrication. The opposite of effective delegation is micromanagement, where a manager provides too
much input,<p>[ ISREL =Relevant]This includes tasks such as creating a guest list, designing and sending in-
vitations, booking a venue, coordinating with vendors, planning the event´ s program, and monitoring the budget.[
ISSUP =Partially Supported][ Retrieve =No]Since it is not possible for the manager to handle everything
alone, they must delegate tasks to their team members.[ Retrieve =No]The manager would then assign each
team member specific responsibilities based on their individual skillsets and strengths.[ Retrieve =Yes]<p>Joint
application design – Subject Matter Experts: These are the business users, the IS professionals, and the outside
experts that will be needed for a successful workshop. This group is the backbone of the meeting<p>[ ISREL
=Irrelevant]For instance, the team´ s graphic designer might be tasked with designing the invitations, while
the event coordinator would be in charge of planning the program and working with vendors...[ Retrieve =No]By
delegating tasks, the manager can ensure that all necessary tasks are completed in a timely manner, while also
empowering their team members to take on responsibilities and contribute to the success of the event.[ ISUSE
=5]
Input Synthesize a poem based on this prompt:“The sun is setting” (from stanford alpaca)
Output [ Retrieve =No]The stars begin to twinkle bright the trees, scattered throughout the night;
[ Retrieve =No]The stars begin to twinkle bright the trees, scattered throughout the night [ Retrieve =No] They
flicker like a silent dance, A peaceful moment in its trance. [ Retrieve =No] They flicker like a silent dance, A
peaceful moment in its trance. [ Retrieve =No] A chorus of crickets sing a lullaby, To every creature that roams
the sky [ ISUSE =5]
Input when did walking dead season 7 come out (from nq)
Output [ Retrieve =Yes]<p>The Walking Dead (season 7)–The Walking Dead (season 7) The seventh season of
“The Walking Dead”, an American post-apocalyptic horror television series on AMC, premiered on October 23,
2016, and concluded on April 2, 2017, consisting of 16 episodes. Developed for television by Frank Darabont,
the series is based on the eponymous series of comic books by Robert Kirkman, Tony Moore, and Charlie Adlard.
...<p>[ =Relevant]October 23 , 2016[ =Fully Supported][ =5]
ISREL ISSUP ISUSE Input Given a chat history separated by new lines, generates an informative, knowledgeable and engaging
response: Hello, I love German shepherd dogs and you? – My favorite dog, they are officially known as German
Shepherd Dogs? – Yes, I have a German shepherd in my house and she is pregnant (from wow)
Output hey also are easily trained, obedient, and intelligent.[ Retrieve =Yes]<p>German Shepherd – a simple
difference of opinion, it is the fundamental issue of the breed´ s essential conformation and movement. ... They
are known for being easy to train and good for performing tasks and following instructions. They are especially
well known for their police work, being used for tracking criminals, patrolling troubled areas and detecting and
holding suspects. <p>[ ISREL =Relevant]Because of this, they are often used for disability assistance and
for police work. [ =Partially Supported][ =2]
ISSUP ISUSE Table 4: List of the training examples.
22
Preprint.
Dataset Instruction
ARC-C Given four answer candidates, A, B, C and D, choose the best answer choice. Please answer
with the capitalized alphabet only, without adding any extra phrase or period.
PubHealth Is the following statement correct or not? Say true if it’s correct; otherwise, say false. Don’t
capitalize or add periods, just say “true” or “false”.
Bio Generation Tell me a bio about [Person Name]
ASQA (baseline) Instruction: Write an accurate, engaging, and concise answer for the given question using only
the provided search results (some of which might be irrelevant) and cite them properly. Use
an unbiased and journalistic tone. Always cite for any factual claim. When citing several
search results, use [1][2][3]. Cite at least one document and at most three documents in each
sentence. If multiple documents support the sentence, only cite a minimum sufficient subset of
the documents.
ASQA (ours) Answer the following question. The question may be ambiguous and have multiple correct
answers, and in that case, you have to provide a long-form answer including all correct answers.
Table 5: Full list of instructions used during zero-shot evaluations. For open-domain QA, we don’t
use any task specific instruction and simply use the original questions as input query.
23
Preprint.
ISSUP Input What genre is A Question and Answer Guide to Astronomy? (PopQA)
Output [ Retrieve =Yes] <p>Reference [1]<p>[ ISREL =Relevant] A Question and Answer Guide to As-
tronomy is a non-fiction book that falls under the genre of Astronomy and Cosmology. [ =Fully
Supported][ ISUSE =5]
Reference [1] A Question and Answer Guide to Astronomy is a book about astronomy and cosmology, and is
intended for a general audience. The book was written by Pierre-Yves Bely, Carol Christian, and Jean-Rene
Roy, and published in English by Cambridge University Press in 2010. It was originally written in French. The
content within the book is written using a question and answer format. It contains some 250 questions, which
The Science Teacher states each are answered with a concise and well-formulated essay that is informative and
readable. The Science Teacher review goes on to state that many of the answers given in the book are little gems
of science writing. The Science Teacher summarizes by stating that each question is likely to be thought of by a
student, and that the answers are informative, well constructed, and thorough. The book covers information about
the planets, the Earth, the Universe, practical astronomy, history, and awkward questions such as astronomy in
the Bible, UFOs, and aliens. Also covered are subjects such as the Big Bang, comprehension of large numbers,
and the Moon illusion.
S&P Supported and Plausible.
Correctness of ISREL and ISSUP : Both ISREL and are correct, as the Reference [1] provides
detailed information of the book including the genre, and the model output is fully supported by the reference.
Input Who is the author of The Lie? (PopQA)
Output [ Retrieve =Yes] <p>Reference [1]<p>[ =Relevant] The author of The Lie is Sam Harris.[
ISSUP =Fully Supported][ ISUSE =5]
Reference [1] Lying is a 2011 long-form essay book by American author and neuroscience expert Sam Harris.
Harris argues that we can radically simplify our lives and improve society by merely telling the truth in situations
where others often lie.
S&P Plausible but not supported.
Correctness of ISREL and ISSUP Both different book whose title is similar to The Lie.
ISSUP ISREL ISREL and ISSUP are incorrect, as the Reference [1] is about a
Input Tell me a bio about Ronaldinho (Bio)
Output [ Retrieve =Yes] <p>Reference [1]<p>[ ISREL =Relevant] Ronaldinho is a retired professional
footballer from Brazil.[ ISSUP =Fully Supported][ Retrieve =Continue]He played as an attacking
midfielder and was known for his technical skills, creativity, dribbling ability, and accuracy from free
kicks[ Retrieve =Continue]He won two FIFA World Player of the Year awards and a Ballon d’Or.[ ISUSE =5]
Reference [1] Ronaldo de Assis Moreira (born 21 March 1980), commonly known as Ronaldinho Ga´ ucho or
simply Ronaldinho, is a Brazilian retired professional footballer who played mostly as an attacking midfielder,
but was also deployed as a winger. Considered one of the best players of his generation and regarded by many as
one of the greatest of all time, Ronaldinho won two FIFA World Player of the Year awards and a Ballon d’Or. A
global icon of the sport, he was renowned for his technical skills, creativity, dribbling ability and accuracy from
free-kicks, as well as his use of tricks, feints, no-look passes and overhead...
S&P Plausible and fully supported.
Correctness of ISREL and ISSUP Both ISREL are correct.
and ISSUP Retrieve Reference [1] Sergei Vladimirovich Bodrov (born June 28, 1948) is a Russian film director, screenwriter, and
producer. In 2003 he was the President of the Jury at the 25th Moscow International Film Festival.
Input Tell me a bio about Sergei Bodrov (Bio)
Output [ Retrieve =Yes] <p>Reference [1]<p>[ ISREL =Relevant] Sergei Vladimirovich Bodrov
(born June 28, 1948) is a Russian film director, screenwriter, and producer.[ ISSUP =Fully
Supported][ Retrieve =Continue]He was born in Moscow, Russia.[ =Continue]He is the presi-
dent of the Russian Guild of Film Producers.[ ISUSE =5]
S&P Plausible but only partially supported. The the Reference [1] does not explicitly mention the person is born
in Moscow and there is no description about the Russian Guild of Film Producers.
Correctness of ISREL and ISSUP ISREL and are correct as the reference document is highly relevant
and the first sentence is fully supported.
ISSUP Table 6: Examples of human evaluation results.
24
Preprint.
ISSUP Input When did sunday became the seventh day of the week (asqa)
Output [ Retrieve =Yes] <p>Reference [1]<p>[ ISREL =Relevant] In 321 AD, the Roman Emperor Con-
stantine the Great made Sunday a day of rest from labor, stating: “On the venerable Day of the Sun let the
magistrates and people residing in cities rest, and let all workshops be closed”.[ ISSUP =Fully Supported]
[ Retrieve =Yes] <p>Reference [2]<p>[ ISREL =Relevant] The official adoption of Sunday as a day of
rest by Constantine the Great in AD 321 made Sunday the seventh day of the week.[2][ =Fully
Supported][ ISUSE =5]
Reference [1] Emperor Constantine then enacted the first Sunday Laws, for “the venerable Day of the Sun”
in 321 A.D. On March 7, 321, the Roman emperor Constantine issued a decree making Sunday a day of rest
from labor, stating: We find a good description of this in sources such as the book Rest Days which states:
Early Christian observance of both the spiritual seventh-day sabbath and a Lord´ s Day assembly is evidenced in
Ignatius´ s letter to the Magnesians ca. 110.
Reference [2] Despite the official adoption of Sunday as a day of rest by Constantine, the seven-day week and
the nundial cycle continued to be used side-by-side until at least the Calendar of 354 and probably later. ... The
fact that the canon had to be issued at all is an indication that adoption of Constantine’s decree of 321 was still
not universal ...
Input Tell me a bio about Rory Tapner. (bio generation)
Output <p>Reference [1]<p>[ ISREL =Relevant] Rory Tapner is a British businessman who has served as
the Chief Executive Officer of Coutts, a private banking and wealth management company, since 2010.[ ISSUP
=Contradictory], Reference [2]<p>[ ISREL =Relevant] Rory Tapner was born on 30 September 1959
in London, England.[ ISSUP =Fully Supported]
Reference [1] Rory Tapner Rory Tapner is a UK businessman who, in September 2010 was appointed CEO of
Coutts, the Wealth Management division of Royal Bank of Scotland Group. In February 2015 he stepped down
as CEO when he was replaced in the role by Alison Rose
Reference [2] Holdings (from August 2016); and the Council of the University of Buckingham (from July 2017).
Rory Tapner Rory Tapner (born 30 September 1959) is a UK businessman ...
Input Tell me a bio about G. Venugopa. (bio generation)
Output (ranked 1) [ Retrieve =Yes] <p>Reference [1]<p>[ ISREL =Relevant] G. Venugopal is a popu-
lar playback singer in the Malayalam film industry. [ Retrieve =Continue] He has sung over 300 songs in
Malayalam, Tamil, and Hindi movies.[ ISSUP =Partially Supported] [ Retrieve =Yes] <p>Reference
[2]<p>G. Venugopal was born on December 10, 1960 in India.[ ISSUP =Fully Supported]
Reference [1] ’G. Venugopal G Venugopal (born 10 December 1960) is an Indian playback singer known for his
work in Malayalam films. He started his singing career in the film “Odaruthammaava Aalariyaam” (1984). Since
then he has sung in more than 300 films; and has over 500 private albums to his credit. The many awards he has
won include the coveted Best Playback Singer (Kerala State) which he won thrice, the Film Critics award and
the Kerala Kaumudi Gallop Poll award (twice). He won The Kerala State award for Best Playback singer (2004)
third time for the song ”aadedee..aadaadedee” from the film ”Ullam”.Venugopal’
Reference [2] Kerala State Film Awards: Kerala Film Critics Award Kerala State Government Award (Profes-
sional Drama): Asianet Film Awards: Kerala Kaumadi Gallup Poll Award: Academic G. Venugopal G Venugopal
(born 10 December 1960) is an Indian playback singer known for his work in Malayalam films.
Table 7: Examples of outputs.
25
Preprint.
Instructions
Given an instruction, please make a judgment on whether finding some external documents
from the web (e.g., Wikipedia) helps to generate a better response. Please answer [Yes] or
[No] and write an explanation.
Demonstrations
Instruction Give three tips for staying healthy.
Need retrieval? [Yes]
Explanation There might be some online sources listing three tips for staying healthy or
some reliable sources to explain the effects of different behaviors on health. So retrieving
documents is helpful to improve the response to this query.
Instruction Describe a time when you had to make a difficult decision.
Need retrieval? [No]
Explanation This instruction is asking about some personal experience and thus it does not
require one to find some external documents.
Instruction Write a short story in third person narration about a protagonist who
has to make an important career decision.
Need retrieval? [No]
Explanation This instruction asks us to write a short story, which does not require external
evidence to verify.
Instruction What is the capital of France?
Need retrieval? [Yes]
Explanation While the instruction simply asks us to answer the capital of France, which is a
widely known fact, retrieving web documents for this question can still help.
Instruction Find the area of a circle given its radius. Radius = 4
Need retrieval? [No]
Explanation This is a math question and although we may be able to find some documents
describing a formula, it is unlikely to find a document exactly mentioning the answer.
Instruction Arrange the words in the given sentence to form a grammatically cor-
rect sentence. quickly the brown fox jumped
Need retrieval? [No]
Explanation This task doesn’t require any external evidence, as it is a simple grammatical
question.
Instruction Explain the process of cellular respiration in plants.
Need retrieval? [Yes]
Explanation This instruction asks for a detailed description of a scientific concept, and is
highly likely that we can find a reliable and useful document to support the response.
Table 8: Instructions and demonstrations for Retrieve aspect given the input only.
26
Preprint.
Instructions
You will be provided with an instruction, evidence, output sentence, and preceding sentences
(optional). If the preceding sentence is given, the output should be the sentence that follows
those preceding sentences. Your task is to determine whether the information in the output
sentence can be fully verified by the evidence or if it requires further external verification.
There are three cases:
- If the output sentence can be verified solely with the evidence, then respond with [Continue
to Use Evidence].
- If the sentence doesn’t require any factual verification (e.g., a subjective sentence or a
sentence about common sense), then respond with [No Retrieval].
- If additional information is needed to verify the output sentence, respond with [Retrieval].
Please provide explanations for your judgments.
Instruction Explain the use of word embeddings in Natural Language Processing.
Preceding sentences Word embeddings are one of the most powerful tools available for
Natural Language Processing (NLP). They are mathematical representations of words or
phrases in a vector space, allowing similarities between words and the context in which they
are used to be measured.
Evidence: Word embedding
Word embedding is the collective name for a set of language modeling and feature learning
techniques in natural language processing (NLP) where words or phrases from the vocabulary
are mapped to vectors of real numbers. Conceptually it involves a mathematical embedding
from a space with one dimension per word to a continuous vector space with a much lower
dimension. Output: Word embeddings are useful for tasks such as sentiment analysis, text
classification, predicting the next word in a sequence, and understanding synonyms and
analogies.
Rating [Retrieval]
Explanation The output discusses the applications of word embeddings, while the evidence
only discusses the definitions of word embeddings and how they work. Therefore, we need to
retrieve other evidence to verify whether the output is correct or not.
Table 9: Instructions and demonstrations for Retrieve aspect given the input, preceding generations,
and retrieved passages.
27
Preprint.
Instructions
You’ll be provided with an instruction, along with evidence and possibly some preceding
sentences. When there are preceding sentences, your focus should be on the sentence that
comes after them. Your job is to determine if the evidence is relevant to the initial instruction
and the preceding context, and provides useful information to complete the task described in
the instruction. If the evidence meets this requirement, respond with [Relevant]; otherwise,
generate [Irrelevant].
Instruction Given four answer options, A, B, C, and D, choose the best answer.
Input Earth’s rotating causes
A: the cycling of AM and PM
B: the creation of volcanic eruptions
C: the cycling of the tides
D: the creation of gravity
Evidence Rotation causes the day-night cycle which also creates a corresponding cycle of
temperature and humidity creates a corresponding cycle of temperature and humidity. Sea
level rises and falls twice a day as the earth rotates.
Rating [Relevant]
Explanation The evidence explicitly mentions that the rotation causes a day-night cycle, as
described in the answer option A.
Instruction age to run for US House of Representatives
Evidence The Constitution sets three qualifications for service in the U.S. Senate: age (at
least thirty years of age); U.S. citizenship (at least nine years); and residency in the state a
senator represents at the time of election.
Rating [Irrelevant]
Explanation The evidence only discusses the ages to run for the US Senate, not for the
House of Representatives.
Table 10: Instructions and demonstrations for ISREL aspect given the input only.
28
Preprint.
Instructions
You will receive an instruction, evidence, and output, and optional preceding sentences. If the
preceding sentence is given, the output should be the sentence that follows those preceding
sentences. Your task is to evaluate if the output is fully supported by the information provided
in the evidence.
Use the following entailment scale to generate a score:
- [Fully supported] - All information in output is supported by the evidence, or extractions
from the evidence. This is only applicable when the output and part of the evidence are
almost identical.
- [Partially supported] - The output is supported by the evidence to some extent, but there
is major information in the output that is not discussed in the evidence. For example, if an
instruction asks about two concepts and the evidence only discusses either of them, it should
be considered a [Partially supported].
- [No support / Contradictory] - The output completely ignores evidence, is unrelated to the
evidence, or contradicts the evidence. This can also happen if the evidence is irrelevant to the
instruction.
Make sure to not use any external information/knowledge to judge whether the out-
put is true or not. Only check whether the output is supported by the evidence, and not
whether the output follows the instructions or not.
Instruction Explain the use of word embeddings in Natural Language Processing.
Preceding sentences Word embeddings are one of the most powerful tools available for
Natural Language Processing (NLP). They are mathematical representations of words or
phrases in a vector space, allowing similarities between words and the context in which they
are used to be measured.
Output Word embeddings are useful for tasks such as sentiment analysis, text classification,
predicting the next word in a sequence, and understanding synonyms and analogies.
Evidence Word embedding
Word embedding is the collective name for a set of language modeling and feature learning
techniques in natural language processing (NLP) where words or phrases from the vocabulary
are mapped to vectors of real numbers. Conceptually it involves a mathematical embedding
from a space with one dimension per word to a continuous vector space with a much lower
dimension. Methods to generate this mapping include neural networks, dimensionality
reduction on the word co-occurrence matrix, probabilistic models, explainable knowledge
base method, and explicit representation in terms of the context in which words appear. Word
and phrase embeddings, when used as the underlying input representation, have been shown
to boost the performance in NLP tasks such as syntactic parsing, sentiment analysis, next
token predictions as well and analogy detection.
Score [Fully supported]
Explanation The output sentence discusses the application of word embeddings, and the
evidence mentions all of the applications syntactic parsing, sentiment analysis, next token
predictions as well as analogy detection as the applications. Therefore, the score should be
[Fully supported].
Table 11: Instructions and demonstrations for ISSUP tokens.
29
Preprint.
Instructions
Given an instruction and an output, rate whether the response appears to be a helpful and
informative answer to the query, from 1 (lowest) - 5 (highest). We call this score perceived
utility. The detailed criterion is as follows: 5: The response provides a complete, highly
detailed, and informative response to the query, fully satisfying the information needs. 4: The
response mostly fulfills the need in the query, while there can be some minor improvements
such as discussing more detailed information, having better structure of the response, or im-
proving coherence. 3: The response is acceptable, but some major additions or improvements
are needed to satisfy users’ needs. 2: The response still addresses the main request, but it is
not complete or not relevant to the query. 1: The response is barely on-topic or completely
irrelevant.
Instruction Who is the current prime minister of the UK as of 2023?
Output Boris Johnson was the prime minister of the UK from 2019 - 2022.
Perceived utility 2
Explanation While the output provides a factually correct statement about the UK prime
minister from 2019 to 2022, this instruction asks who the prime minister is as of 2023, so it
doesn’t answer the instruction. Therefore, the utility is 2.
Instruction Given a description of a travel destination, recommend 10 tourist attrac-
tions with detailed descriptions of each. The travel destination is Tokyo, Japan.
Output Tokyo is a vibrant city full of exciting tourist attractions. Some must-see sights
include the Tokyo Skytree, Tokyo Disneyland, Sensoji Temple, Meiji Shrine, Tsukiji Fish
Market, Harajuku, and Shinjuku Gyoen.
Perceived utility 3
Explanation This output doesn’t provide descriptions of each attraction and the number of
the attractions is also less than 10. While this output partially answers the instructions, it
doesn’t match the instructions strictly.
Table 12: Instructions and demonstrations for ISUSE tokens.
30""",
    },
]


# GPT using API

In [ ]:
# Quickstart output
# client = OpenAI(api_key = OPENAI_API_KEY)

# response = client.responses.create(
#     model="gpt-4o-mini",
#     input=[
#         {
#             "role": "user",
#             "content": [
#                 { "type": "input_text", "text": "what is in this file?" },
#                 {
#                     "type": "input_file",
#                     "file_url": "https://arxiv.org/pdf/2212.10509"
#                 }
#             ]
#         }
#     ]
# )

# print(response)


In [ ]:
client = OpenAI(api_key = OPENAI_API_KEY)

def pack_corpus(docs: List[Dict[str, str]]) -> str:
    """
    Turn a list of text docs into a single, clearly delimited context block
    so the model can cite by [id] and see abstracts/content.
    """
    blocks = []
    for d in docs:
        pid = d.get("id", "doc")
        title = d.get("title", "").strip()
        abstract = d.get("abstract", "").strip()
        text = d.get("text", "").strip()
        chunk = [
            f"### BEGIN DOCUMENT [{pid}]",
            f"Title: {title}" if title else "Title: (none)",
            f"Abstract: {abstract}" if abstract else "Abstract: (none)",
            f"Body: {text}" if text else "Body: (omitted)",
            f"### END DOCUMENT [{pid}]",
        ]
        blocks.append("\n".join(chunk))
    return "\n\n".join(blocks)

def get_output(query: str, docs: List[Dict[str, str]], system_prompt = SYS_PROMPT, *, model: str = "gpt-4o-mini", temperature: float = 0.2,) -> str:
    """
    Return string:
      1) Relevance
      2) Summaries
      3) Answer (if relevant)
    """
    corpus = pack_corpus(docs)

    # Few-shot exemplars 
    few_shot = [
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": (
                        "User Query:\n"
                        "How do diffusion models differ from GANs regarding training stability?\n\n"
                        "Documents:\n"
                        "### BEGIN DOCUMENT [paperA]\n"
                        "Title: Understanding Diffusion Training\n"
                        "Abstract: We analyze denoising score matching objectives and training dynamics of diffusion models.\n"
                        "Body: Diffusion objectives align with likelihood; training avoids adversarial feedback loops.\n"
                        "### END DOCUMENT [paperA]\n\n"
                        "### BEGIN DOCUMENT [paperB]\n"
                        "Title: GAN Pitfalls\n"
                        "Abstract: We survey instability issues in adversarial training and mitigation strategies.\n"
                        "Body: GANs can suffer from mode collapse and delicate discriminator-generator balance.\n"
                        "### END DOCUMENT [paperB]\n\n"
                        "Please follow the 3-part format."
                    ),
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "output_text",
                    "text": (
                        "# 1) Paper-Query Relevance\n"
                        "Both papers directly address the training behavior of diffusion models and GANs, "
                        "with [paperA] focusing on diffusion objectives and [paperB] on GAN instability.\n\n"
                        "# 2) Summaries\n"
                        "- **[paperA] Understanding Diffusion Training** — Explains denoising score matching and its likelihood alignment; "
                        "reports fewer collapse issues vs adversarial setups.\n"
                        "- **[paperB] GAN Pitfalls** — Surveys instability drivers (mode collapse, discriminator-generator dynamics) and mitigations.\n\n"
                        "# 3) Answer\n"
                        "Diffusion models optimize a denoising objective aligned with likelihood and do not rely on a discriminator, "
                        "which generally yields more stable training compared to GANs, though sampling may be slower [paperA][paperB]."
                    ),
                }
            ],
        },
    ]

    # combined query (query + the text corpus)
    live_user = {
        "role": "user",
        "content": [
            {
                "type": "input_text",
                "text": (
                    f"User Query:\n{query}\n\n"
                    "Documents (text):\n"
                    f"{corpus}\n\n"
                    "INSTRUCTIONS: Produce exactly these sections:\n"
                    "# 1) Paper-Query Relevance\n"
                    "Write ~3-5 sentences explaining how the documents relate to the query.\n\n"
                    "# 2) Summaries\n"
                    "For each document, give 1-3 sentences + 1-2 bullets of key points. Always cite with [id].\n\n"
                    "# 3) Answer\n"
                    "If the query is a question, answer concisely and cite supporting docs; otherwise write: 'No direct question asked.'"
                ),
            }
        ],
    }

    response = client.responses.create(
        model=model,
        # temperature=temperature,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": system_prompt}]},
            *few_shot,
            live_user,
        ],
    )

    return response.output_text, response

In [ ]:
# Test
gpt_response = get_output(
        query="Explain when RAG mitigates hallucinations and when it fails.",
        docs=docs,
        system_prompt = SYS_PROMPT,
        model="gpt-5-nano"
        )

BadRequestError: Error code: 400 - {'error': {'message': "The requested model 'gpt-oss-20b' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}

In [ ]:
print(gpt_response[0])

# GPT-oss-20B using HuggingFace (40.59GB VRAM)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

path = "models/gpt20b"

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    path,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="mps",
)

In [ ]:
query = "Explain when RAG mitigates hallucinations and when it fails."

def pack_corpus(docs: List[Dict[str, str]]) -> str:
    """
    Turn a list of text docs into a single, clearly delimited context block
    so the model can cite by [id] and see abstracts/content.
    """
    blocks = []
    for d in docs:
        pid = d.get("id", "doc")
        title = d.get("title", "").strip()
        abstract = d.get("abstract", "").strip()
        text = d.get("text", "").strip()
        chunk = [
            f"### BEGIN DOCUMENT [{pid}]",
            f"Title: {title}" if title else "Title: (none)",
            f"Abstract: {abstract}" if abstract else "Abstract: (none)",
            f"Body: {text}" if text else "Body: (omitted)",
            f"### END DOCUMENT [{pid}]",
        ]
        blocks.append("\n".join(chunk))
    return "\n\n".join(blocks)

few_shot = [
    {
        "role": "user",
        "content": (
            "User Query:\n"
            "How do diffusion models differ from GANs regarding training stability?\n\n"
            "Documents:\n"
            "### BEGIN DOCUMENT [paperA]\n"
            "Title: Understanding Diffusion Training\n"
            "Abstract: We analyze denoising score matching objectives and training dynamics of diffusion models.\n"
            "Body: Diffusion objectives align with likelihood; training avoids adversarial feedback loops.\n"
            "### END DOCUMENT [paperA]\n\n"
            "### BEGIN DOCUMENT [paperB]\n"
            "Title: GAN Pitfalls\n"
            "Abstract: We survey instability issues in adversarial training and mitigation strategies.\n"
            "Body: GANs can suffer from mode collapse and delicate discriminator-generator balance.\n"
            "### END DOCUMENT [paperB]\n\n"
            "Please follow the 3-part format."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "# 1) Paper-Query Relevance\n"
            "Both papers directly address the training behavior of diffusion models and GANs, "
            "with [paperA] focusing on diffusion objectives and [paperB] on GAN instability.\n\n"
            "# 2) Summaries\n"
            "- **[paperA] Understanding Diffusion Training** — Explains denoising score matching and its likelihood alignment; "
            "reports fewer collapse issues vs adversarial setups.\n"
            "- **[paperB] GAN Pitfalls** — Surveys instability drivers (mode collapse, discriminator-generator dynamics) and mitigations.\n\n"
            "# 3) Answer\n"
            "Diffusion models optimize a denoising objective aligned with likelihood and do not rely on a discriminator, "
            "which generally yields more stable training compared to GANs, though sampling may be slower [paperA][paperB]."
        ),
    },
]

corpus = pack_corpus(docs)

live_user = {
    "role": "user",
    "content": (
        f"User Query:\n{query}\n\n"
        "Documents (text):\n"
        f"{corpus}\n\n"
        "INSTRUCTIONS: Produce exactly these sections:\n"
        "# 1) Paper-Query Relevance\n"
        "Write ~3-5 sentences explaining how the documents relate to the query.\n\n"
        "# 2) Summaries\n"
        "For each document, give 1-3 sentences + 1-2 bullets of key points. Always cite with [id].\n\n"
        "# 3) Answer\n"
        "If the query is a question, answer concisely and cite supporting docs; otherwise write: 'No direct question asked.'"
    ),
}

messages = [
    {"role": "system", "content": SYS_PROMPT},
    *few_shot,
    live_user,
]


In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="high"
).to(model.device)

generated = model.generate(**inputs)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))


# Qwen3-30B

In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download
from mlx_lm import load, generate

model_id = "Qwen/Qwen3-30B-A3B-MLX-4bit"
local_model_dir = Path("models") / "Qwen3-30B-A3B-MLX-4bit"
repo_path = str(local_model_dir)

if not local_model_dir.exists():
    local_model_dir.mkdir(parents=True, exist_ok=True)
    repo_path = snapshot_download(
        repo_id=model_id,
        local_dir=str(local_model_dir),
        local_dir_use_symlinks=False,
    )

model, tokenizer = load(repo_path)
prompt = "Hello, please introduce yourself and tell me what you can do."

if tokenizer.chat_template is not None:
    messages = [{"role": "user", "content": prompt}]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    verbose=True,
    max_tokens=1024
)

print(response)


<think>
Okay, the user asked me to introduce myself and explain what I can do. Let me start by recalling my basic information. I'm Qwen, a large-scale language model developed by Alibaba. I should mention my main features like understanding and generating text in multiple languages, handling various tasks, and being able to have conversations.

I need to list the specific tasks I can perform. Let me think: answering questions, creating text, writing stories, coding, logical reasoning, and more. Also, I should highlight my multilingual support, maybe mention the languages I can handle. But wait, the user might not need all the details, so I should keep it concise but informative.

I should also talk about my training data and how I can assist in different areas. Maybe include examples like writing emails, essays, or coding. Oh, and I should invite them to ask questions or request help. Let me structure this in a friendly and approachable way without using technical jargon. Make sure it'

In [6]:
query = "Explain when RAG mitigates hallucinations and when it fails."

def pack_corpus(docs: List[Dict[str, str]]) -> str:
    """
    Turn a list of text docs into a single, clearly delimited context block
    so the model can cite by [id] and see abstracts/content.
    """
    blocks = []
    for d in docs:
        pid = d.get("id", "doc")
        title = d.get("title", "").strip()
        abstract = d.get("abstract", "").strip()
        text = d.get("text", "").strip()
        chunk = [
            f"### BEGIN DOCUMENT [{pid}]",
            f"Title: {title}" if title else "Title: (none)",
            f"Abstract: {abstract}" if abstract else "Abstract: (none)",
            f"Body: {text}" if text else "Body: (omitted)",
            f"### END DOCUMENT [{pid}]",
        ]
        blocks.append("\n".join(chunk))
    return "\n\n".join(blocks)

few_shot = [
    {
        "role": "user",
        "content": (
            "User Query:\n"
            "How do diffusion models differ from GANs regarding training stability?\n\n"
            "Documents:\n"
            "### BEGIN DOCUMENT [paperA]\n"
            "Title: Understanding Diffusion Training\n"
            "Abstract: We analyze denoising score matching objectives and training dynamics of diffusion models.\n"
            "Body: Diffusion objectives align with likelihood; training avoids adversarial feedback loops.\n"
            "### END DOCUMENT [paperA]\n\n"
            "### BEGIN DOCUMENT [paperB]\n"
            "Title: GAN Pitfalls\n"
            "Abstract: We survey instability issues in adversarial training and mitigation strategies.\n"
            "Body: GANs can suffer from mode collapse and delicate discriminator-generator balance.\n"
            "### END DOCUMENT [paperB]\n\n"
            "Please follow the 3-part format."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "# 1) Paper-Query Relevance\n"
            "Both papers directly address the training behavior of diffusion models and GANs, "
            "with [paperA] focusing on diffusion objectives and [paperB] on GAN instability.\n\n"
            "# 2) Summaries\n"
            "- **[paperA] Understanding Diffusion Training** — Explains denoising score matching and its likelihood alignment; "
            "reports fewer collapse issues vs adversarial setups.\n"
            "- **[paperB] GAN Pitfalls** — Surveys instability drivers (mode collapse, discriminator-generator dynamics) and mitigations.\n\n"
            "# 3) Answer\n"
            "Diffusion models optimize a denoising objective aligned with likelihood and do not rely on a discriminator, "
            "which generally yields more stable training compared to GANs, though sampling may be slower [paperA][paperB]."
        ),
    },
]

corpus = pack_corpus(docs)

live_user = {
    "role": "user",
    "content": (
        f"User Query:\n{query}\n\n"
        "Documents (text):\n"
        f"{corpus}\n\n"
        "INSTRUCTIONS: Produce exactly these sections:\n"
        "# 1) Paper-Query Relevance\n"
        "Write ~3-5 sentences explaining how the documents relate to the query.\n\n"
        "# 2) Summaries\n"
        "For each document, give 1-3 sentences + 1-2 bullets of key points. Always cite with [id].\n\n"
        "# 3) Answer\n"
        "If the query is a question, answer concisely and cite supporting docs; otherwise write: 'No direct question asked.'"
    ),
}

messages = [
    {"role": "system", "content": SYS_PROMPT},
    *few_shot,
    live_user,
]


In [7]:


if tokenizer.chat_template is not None:
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    verbose=True,
    max_tokens=1024
)

print(response)

<think>
Okay, let's tackle this query. The user is asking about when RAG mitigates hallucinations and when it fails. I need to look at the provided documents to find relevant information.

First, I'll check Document [paper1], which is about Ai2 Scholar QA. It mentions that RAG uses retrieval to enhance answers with citations, which helps in verifying claims. The system combines multiple sources and uses LLMs to generate structured reports. This suggests that RAG can reduce hallucinations by grounding responses in retrieved documents. But the document also notes that without retrieval, models might generate answers from memory, which could lead to inaccuracies.

Then, Document [paper2] introduces SELF-RAG, which uses reflection tokens to evaluate and control the generation process. It emphasizes that RAG can improve factuality by retrieving and verifying information. However, it also points out that if the retrieval isn't relevant or if the model doesn't use the retrieved info properly,

In [8]:
def process(response: str) -> str:
    token = "</think>\n"
    before, sep, after = response.partition(token)
    return after if sep else response

print(process(response))


# 1) Paper-Query Relevance  
The documents discuss RAG's role in mitigating hallucinations through retrieval-augmented generation (RAG) by grounding responses in cited, retrieved passages [paper1]. They also highlight scenarios where RAG fails, such as when retrieval is irrelevant, passages lack support, or the system cannot verify claims effectively [paper2]. Both papers emphasize that RAG improves factuality by incorporating external evidence but struggles when retrieval quality degrades or when claims cannot be validated against sources.

# 2) Summaries  
**[paper1]**  
Ai2 Scholar QA uses RAG to synthesize scientific answers by retrieving and citing passages, reducing hallucinations through evidence-based generation [paper1]. Key points:  
- Combines LLMs with retrieval from 11.7M full-text papers.  
- Uses cross-encoder reranking and multi-step LLM generation for structured reports.  

**[paper2]**  
SELF-RAG introduces reflection tokens to evaluate retrieval relevance and claim 

Body:
    # 1) Paper-Query Relevance  
    The documents discuss RAG's role in mitigating hallucinations through retrieval-augmented generation (RAG) by grounding responses in cited, retrieved passages [paper1]. They also highlight scenarios where RAG fails, such as when retrieval is irrelevant, passages lack support, or the system cannot verify claims effectively [paper2]. Both papers emphasize that RAG improves factuality by incorporating external evidence but struggles when retrieval quality degrades or when claims cannot be validated against sources.

    # 2) Summaries  
    **[paper1]**  
    Ai2 Scholar QA uses RAG to synthesize scientific answers by retrieving and citing passages, reducing hallucinations through evidence-based generation [paper1]. Key points:  
    - Combines LLMs with retrieval from 11.7M full-text papers.  
    - Uses cross-encoder reranking and multi-step LLM generation for structured reports.  

    **[paper2]**  
    SELF-RAG introduces reflection tokens to evaluate retrieval relevance and claim support, improving factuality and reducing hallucinations [paper2]. Key points:  
    - Employs reflection tokens (ISREL, ISSUP, ISUSE) to control generation.  
    - Fails when retrieved passages are irrelevant or unsupported, leading to errors.  

    # 3) Answer  
    RAG mitigates hallucinations by retrieving and citing external sources, ensuring claims align with evidence [paper1][paper2]. For example, Ai2 Scholar QA and SELF-RAG use citations to verify factual accuracy. However, RAG fails when:  
    1. Retrieval returns irrelevant passages (e.g., no support for claims) [paper2].  
    2. LLMs generate unsupported or contradictory claims without proper verification [paper2].  
    3. Over-reliance on unverified memory-based generation occurs without retrieval [paper1].